# FINAL

This notebook merges the current pair-level, control, and network-topology analyses into one runnable analysis notebook for the 906-neuron MICrONS structure-function cohort.

The assignment question is: construct the functional network from the imaging data, compare it with the structural network, ask whether directly connected neurons are strongly correlated, and then perform network analysis to evaluate whether the two networks have similar topology.

The notebook is organized around seven analysis questions. The map below defines those questions before any data are loaded or any code is run, so the later sections can use the short hypothesis labels without ambiguity.

Person 4's 93-neuron sanity check and V1/RL/AL exploratory analysis are still pending. The final synthesis is therefore provisional until that section is appended.


## 0. Hypothesis Map and Rationale

The hypotheses are not separate projects; they are a staged answer to the same structure-function question. H1 tests the direct assignment question. H2 and H3 ask whether the structural edge details add information beyond binary connectedness. H4-H6 test whether the H1 result survives the most obvious confounds. H7 moves from pair-level comparisons to network topology.

| Hypothesis | Analysis question | Why we consider it | Link to the overarching question | Main readout |
|---|---|---|---|---|
| H1. Connected pairs | Are directly connected neuron pairs more functionally correlated than unconnected pairs? | This is the central assignment question and the most direct structure-function comparison. | If anatomy carries functional information, connected pairs should have higher signal correlation on average. | Connected minus unconnected mean `F_corr`, confidence interval, effect size, and neuron-identity permutation null. |
| H2. Synapse strength | Among connected pairs, do stronger synaptic connections have higher functional correlation? | A binary edge may be too coarse; synapse size asks whether edge weight matters. | Tests whether the structural network predicts function in a graded way rather than only as connected versus unconnected. | Spearman correlation between summed synapse size and `F_corr`, plus strength quartile means. |
| H3. Reciprocity | Are bidirectional pairs more functionally similar than unidirectional pairs? | The structural graph is directed, while the functional graph is symmetric; reciprocity is one way to bridge that mismatch. | Tests whether richer directed structural motifs add functional similarity beyond the presence of any edge. | Mean `F_corr` for `none`, `uni`, and `bi` pairs, with pairwise and permutation tests. |
| H4. Distance control | Does the connected-pair effect remain after accounting for soma-soma distance? | Nearby neurons are more likely to be connected and can also be more functionally similar, so distance is the major geometric confound. | Separates direct structure-function alignment from shared spatial organization. | Distance-matched H1 retest, within-distance-bin tests, and logistic control for distance. |
| H5. Composition controls | Does the effect remain within and across area, layer, and cell-type groupings? | Area, layer, and cell type can create apparent functional similarity even without direct connectivity. | Tests whether H1 is a connectivity effect rather than only a composition effect. | Stratified H1 contrasts and joint logistic model with distance plus composition indicators. |
| H6. Orientation similarity | Is connectivity better explained by visual tuning similarity, especially orientation preference? | In visual cortex, like-to-like wiring may reflect tuning similarity; orientation is the most interpretable available tuning axis. | Tests whether signal correlation is just a proxy for orientation co-tuning or captures broader shared response structure. | Orientation-similarity contrast and joint model with `F_corr`, `ori_sim`, and distance. |
| H7. Network topology / hub coupling | Do structurally central neurons also look functionally central? | The prompt asks for network analysis and topology comparison, not only pairwise edge tests. | Tests whether structure-function similarity extends from individual pairs to node-level organization. | Structural degree/strength versus mean functional coupling, partial Spearman tests, and thresholded topology sensitivity. |

The pending Person 4 section is best treated as validation and scope extension rather than a new main hypothesis: it will add the 93-neuron same-scan sanity check and the V1/RL/AL exploratory comparison after the current H1-H7 path.


## 1. Shared Setup, Data Loading, and Conventions

All downstream analyses use this shared setup. The goal is to make the final notebook runnable from a fresh kernel without relying on hidden state from the original individual notebooks.

The structural convention is fixed once here: `C[i, j]` means a directed synaptic edge from presynaptic neuron `i` to postsynaptic neuron `j`. Functional similarity is measured with signal correlation across the shared imaging response matrix. Pair-level tests use unordered pairs `i < j`, because the functional correlation matrix is symmetric, while node-level topology tests later return to directed structural degree and strength.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from scipy import sparse, stats
from scipy.stats import mannwhitneyu, spearmanr, pearsonr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import statsmodels.api as sm
import networkx as nx

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = Path.cwd()
DATA = ROOT / 'Data' if (ROOT / 'Data').exists() else ROOT / 'data'
DATA_DIR = DATA
CACHE_DIR = DATA / 'cache'
CACHE = CACHE_DIR
EXPORTS_DIR = DATA / 'exports'
SYN_CSV = DATA / '1718' / 'raw' / 'synapses_matched.csv'

FIG_DIR = ROOT / 'outputs' / 'figures'
TBL_DIR = ROOT / 'outputs' / 'tables'
OUT_FIG = FIG_DIR
OUT_TBL = TBL_DIR
for path in [FIG_DIR, TBL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PAL_CONN = {'none': '#aab7c4', 'uni': '#4E79A7', 'bi': '#C8453B'}
PAL_AREA = {'V1': '#4E79A7', 'RL': '#F28E2B', 'AL': '#59A14F'}
ORDER_CONN = ['none', 'uni', 'bi']
RAMP_SEQ = sns.color_palette('mako', as_cmap=True)
RAMP_FORE = sns.color_palette('rocket', as_cmap=True)
RAMP_DIV = plt.get_cmap('RdBu_r')
RNG_SEED = 0

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'root       : {ROOT}')
print(f'data       : {DATA}')
print(f'figures out: {FIG_DIR}')
print(f'tables out : {TBL_DIR}')


## 2. Shared Cohort, Functional Matrix, and Structural Matrix

The 906-neuron cohort and the functional response matrix are loaded once. The structural matrix uses `Data/exports/G_906_edges.csv`, which matches the raw matched synapse table after filtering to this cohort and is simpler to use consistently in the merged notebook.

This section also prints the key sanity checks that all later hypotheses depend on: neuron count, functional response shape, valid functional rows, brain-area composition, directed structural edge count, directed density, and the number of finite off-diagonal functional correlations.


In [ ]:
# Cohort and functional responses
matched = pd.read_pickle(CACHE_DIR / 'matched_906.pkl').reset_index(drop=True)
N = len(matched)
F = np.load(CACHE_DIR / 'F_906.npy')
valid_path = CACHE_DIR / 'F_906_valid.npy'
valid = np.load(valid_path).astype(bool) if valid_path.exists() else np.isfinite(F).all(axis=1)

if F.shape[0] != N:
    raise ValueError(f'F has {F.shape[0]} rows but matched cohort has {N} neurons')
if valid.shape[0] != N:
    raise ValueError(f'valid mask has {valid.shape[0]} rows but matched cohort has {N} neurons')

# Canonical structural input for this final notebook: exported 906 directed edge table.
edges = pd.read_csv(EXPORTS_DIR / 'G_906_edges.csv')
required_edge_cols = {'pre_neuron_id', 'post_neuron_id', 'synapse_size'}
missing = required_edge_cols.difference(edges.columns)
if missing:
    raise ValueError(f'G_906_edges.csv is missing required columns: {sorted(missing)}')

pt_to_idx = pd.Series(np.arange(N), index=matched['pt_root_id'].astype('int64'))
pre_id = edges['pre_neuron_id'].astype('int64')
post_id = edges['post_neuron_id'].astype('int64')
valid_edge = pre_id.isin(pt_to_idx.index) & post_id.isin(pt_to_idx.index) & (pre_id != post_id)
edges_in = edges.loc[valid_edge].copy()

rows = pt_to_idx.loc[edges_in['pre_neuron_id'].astype('int64')].to_numpy()
cols = pt_to_idx.loc[edges_in['post_neuron_id'].astype('int64')].to_numpy()
data = edges_in['synapse_size'].to_numpy(dtype=np.float64)
C = sparse.coo_matrix((data, (rows, cols)), shape=(N, N)).tocsr()
C.sum_duplicates()
C_strength = C
A_struct = C_strength != 0

F_corr = np.corrcoef(F)
off_diag = ~np.eye(N, dtype=bool)
finite = np.isfinite(F_corr) & off_diag

area_counts = matched['brain_area'].value_counts(dropna=False).to_dict()
directed_density = C.nnz / (N * (N - 1))
print(f'neurons                  : {N:,}')
print(f'functional response shape: {F.shape}')
print(f'valid functional neurons : {valid.sum():,} / {N:,}')
print(f'brain areas              : {area_counts}')
print(f'directed structural edges: {C.nnz:,}')
print(f'directed density         : {directed_density:.4f}')
print(f'finite off-diagonal Fcorr: {finite.sum():,} / {off_diag.sum():,}')


## 3. Shared Pair Table for Pair-Level Analyses

This table is the backbone of the pair-level hypotheses defined above. It contains one row per unordered neuron pair and combines functional similarity with structural edge labels, synaptic-strength summaries, distance, coarse composition, and orientation similarity.

The same table feeds H1-H6 so that the analyses are comparable. H1 uses `connected` and `f_corr`; H2 adds synaptic strength; H3 uses `conn_type`; H4 uses `dist_um`; H5 uses area, layer, and cell-type indicators; H6 uses `ori_sim` and `gOSI_min`. Keeping this table shared prevents small differences in filtering or matrix alignment from creating inconsistent results across sections.


In [ ]:
# Shared unordered pair table for H1-H6.
# Matrix convention throughout: C[i, j] is the edge from presynaptic neuron i to postsynaptic neuron j.
C_dense = C.toarray()
A_pre = C_dense > 0
A_post = C_dense.T > 0
A_und = A_pre | A_post
np.fill_diagonal(A_und, False)
bidir = A_pre & A_post

xyz = matched[['pt_position_x', 'pt_position_y', 'pt_position_z']].to_numpy(dtype=float)
ori = np.deg2rad(matched['pref_ori'].to_numpy(dtype=float))
gOSI = matched['gOSI'].to_numpy(dtype=float)
area = matched['brain_area'].astype(str).to_numpy()
layer = matched['layer'].astype(str).to_numpy()
ctype = matched['cell_type'].astype(str).to_numpy()

iu, ju = np.triu_indices(N, k=1)
fc_pair = F_corr[iu, ju]
keep = np.isfinite(fc_pair)
iu, ju = iu[keep], ju[keep]
area_i, area_j = area[iu], area[ju]

syn_ij = C_dense[iu, ju]
syn_ji = C_dense[ju, iu]
pairs = pd.DataFrame({
    'i': iu,
    'j': ju,
    'pre_to_post_ij': A_pre[iu, ju],
    'pre_to_post_ji': A_pre[ju, iu],
    'f_corr': F_corr[iu, ju],
    'syn_ij': syn_ij,
    'syn_ji': syn_ji,
    'syn_size_max': np.maximum(syn_ij, syn_ji),
    'syn_size_sum': syn_ij + syn_ji,
    'connected': A_und[iu, ju],
    'conn_type': np.where(bidir[iu, ju], 'bi', np.where(A_und[iu, ju], 'uni', 'none')),
    'dist_um': np.linalg.norm(xyz[iu] - xyz[ju], axis=1),
    'same_area': area_i == area_j,
    'same_layer': layer[iu] == layer[ju],
    'same_celltype': ctype[iu] == ctype[ju],
    'area_pair': [f'{a}-{b}' if a <= b else f'{b}-{a}' for a, b in zip(area_i, area_j)],
    'ori_sim': np.cos(2.0 * (ori[iu] - ori[ju])),
    'gOSI_min': np.minimum(gOSI[iu], gOSI[ju]),
})

n_total_possible = N * (N - 1) // 2
print(f'unordered pairs possible : {n_total_possible:,}')
print(f'pairs with finite F_corr : {len(pairs):,}')
print(f'connected pairs          : {pairs.connected.sum():,}')
print(f'unconnected pairs        : {(~pairs.connected).sum():,}')
print('conn_type counts         :', pairs['conn_type'].value_counts().reindex(ORDER_CONN).to_dict())
print()
print('mean F_corr by connection class:')
print(pairs.groupby('conn_type')['f_corr'].agg(['count', 'mean', 'std']).reindex(ORDER_CONN))
pairs.head()


## 4. H1. Directly Connected Neurons Have Higher Functional Similarity

H1 is the main answer to the assignment question. The structural network says whether a pair of neurons has a direct synaptic connection; the functional network gives the pair's signal correlation. If structural connectivity carries functional information, connected pairs should have higher `F_corr` than unconnected pairs on average.

The first analysis reports the raw connected-versus-unconnected contrast, including the mean shift, bootstrap confidence interval, Mann-Whitney test, and Cohen's d. The second analysis is the more important inferential check: a neuron-identity permutation null that keeps the structural graph and functional values fixed but randomizes which neuron identities are paired. This matters because pair rows are not independent; each neuron appears in many pairs.


In [ ]:
def boot_mean_diff(a, b, n_boot=5000, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    na, nb = len(a), len(b)
    for k in range(n_boot):
        diffs[k] = rng.choice(a, na, replace=True).mean() - rng.choice(b, nb, replace=True).mean()
    return diffs.mean(), np.percentile(diffs, [2.5, 97.5])

def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp

f_none = pairs.loc[pairs.conn_type == 'none', 'f_corr'].to_numpy()
f_uni  = pairs.loc[pairs.conn_type == 'uni',  'f_corr'].to_numpy()
f_bi   = pairs.loc[pairs.conn_type == 'bi',   'f_corr'].to_numpy()
f_conn = pairs.loc[pairs.connected,           'f_corr'].to_numpy()

U_h1, p_h1     = mannwhitneyu(f_conn, f_none, alternative='greater')
md_h1, ci_h1   = boot_mean_diff(f_conn, f_none)
d_h1           = cohens_d(f_conn, f_none)
median_diff_h1 = float(np.median(f_conn) - np.median(f_none))

print(f'H1 - connected (n={len(f_conn):,}) vs unconnected (n={len(f_none):,})')
print(f'  delta-mean   (conn - none) = {md_h1:+.4f}    95% CI [{ci_h1[0]:+.4f}, {ci_h1[1]:+.4f}]')
print(f'  delta-median (conn - none) = {median_diff_h1:+.4f}')
print(f'  Mann-Whitney U             = {U_h1:.3e}    one-sided p = {p_h1:.3e}')
print(f"  Cohen's d                  = {d_h1:+.3f}    (small effect by Cohen's conventions)")
print('  NOTE: the MW p-value treats pairs as independent, which they are not;')
print('        a neuron-level permutation null is computed in the next section.')

H1_RESULT = {
    'hypothesis':     'H1',
    'description':    'connected > unconnected (signal F_corr)',
    'test_direction': 'one-sided (connected > unconnected)',
    'n_connected':    int(len(f_conn)),
    'n_unconnected':  int(len(f_none)),
    'mean_diff':      float(md_h1),
    'median_diff':    median_diff_h1,
    'ci_lo':          float(ci_h1[0]),
    'ci_hi':          float(ci_h1[1]),
    'cohens_d':       float(d_h1),
    'mannwhitney_U':  float(U_h1),
    'p_one_sided':    float(p_h1),
}

In [ ]:
# ---- neuron-identity permutation null for H1 ----
# The structural undirected adjacency A_und (computed in section 4) is held fixed.
# For each permutation pi we re-read each pair's connection status as A_und[pi[i], pi[j]],
# recompute Delta-mean(conn - none) using the original f_corr values, and tally how
# often that exceeds the observed Delta-mean.

iu_all, ju_all = np.triu_indices(N, k=1)
fmask = np.isfinite(F_corr[iu_all, ju_all])
iu_all = iu_all[fmask]
ju_all = ju_all[fmask]
f_pairs_arr = F_corr[iu_all, ju_all]

# A_und was implicitly built in section 4; reconstruct it here so this cell is self-contained
A_und = (C_dense > 0) | (C_dense.T > 0)
np.fill_diagonal(A_und, False)

obs_diff = float(f_conn.mean() - f_none.mean())

N_PERM = 10000
rng = np.random.default_rng(RNG_SEED)
perm_diffs = np.empty(N_PERM)
for k in range(N_PERM):
    pi = rng.permutation(N)
    conn_k = A_und[pi[iu_all], pi[ju_all]]
    perm_diffs[k] = f_pairs_arr[conn_k].mean() - f_pairs_arr[~conn_k].mean()

n_exceed = int((perm_diffs >= obs_diff).sum())
p_perm   = (n_exceed + 1) / (N_PERM + 1)   # one-sided, with the standard +1/+1 correction
null_mu  = float(perm_diffs.mean())
null_sd  = float(perm_diffs.std(ddof=1))
null_lo, null_hi = np.percentile(perm_diffs, [2.5, 97.5])
z_obs = (obs_diff - null_mu) / null_sd if null_sd > 0 else float('inf')

print(f'H1 neuron-identity permutation null (n_perm = {N_PERM:,})')
print(f'  observed delta-mean    = {obs_diff:+.4f}')
print(f'  null mean +/- std      = {null_mu:+.4f} +/- {null_sd:.4f}')
print(f'  null 95% range         = [{null_lo:+.4f}, {null_hi:+.4f}]')
print(f'  observed sits at z     = {z_obs:+.2f}  std deviations from null mean')
print(f'  one-sided perm p-value = {p_perm:.4e}   (#perms >= observed: {n_exceed}/{N_PERM})')

H1_RESULT['perm_n']             = int(N_PERM)
H1_RESULT['perm_null_mean']     = null_mu
H1_RESULT['perm_null_std']      = null_sd
H1_RESULT['perm_z']             = float(z_obs)
H1_RESULT['perm_p_one_sided']   = float(p_perm)

### H1 Conclusion

H1 is the strongest current result and should anchor the notebook's story. In the current 906-neuron run, connected pairs show a clear upward shift in signal correlation relative to unconnected pairs. The effect is modest in absolute size, but it is robust under the neuron-identity permutation null, which is the appropriate check for pair non-independence.

The scientific interpretation is not that direct anatomy determines the response similarity of any single neuron pair. The result is a population-level statement: direct synaptic connectivity enriches for functional similarity. That is exactly the first-order structure-function comparison the assignment asks for, and the later hypotheses should be read as tests of how much weight this result can carry after adding edge weights, directionality, spatial controls, composition controls, tuning controls, and topology-level comparisons.


## 5. H2. Synapse Strength Gradient

H2 asks whether the H1 effect is only a binary edge effect or whether stronger anatomical connections also predict stronger functional similarity. This is important because a structural network can be represented either as an unweighted adjacency matrix or as a weighted synaptic graph.

The primary structural weight here is summed synapse size across the two directions of an unordered pair. Spearman correlation is used because synapse size is heavy-tailed and the question is whether stronger pairs tend to rank higher in functional similarity. The log-strength Pearson check and strength-quartile means help translate the rank result into a more interpretable effect-size scale.


In [ ]:
def boot_spearman_ci(x, y, n_boot=2000, seed=RNG_SEED):
    """95% percentile bootstrap CI for Spearman rho. Pairs (x, y) are resampled
    jointly so the rank-relation within each resample is preserved."""
    rng = np.random.default_rng(seed)
    n = len(x)
    x = np.asarray(x); y = np.asarray(y)
    rhos = np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, n, n)
        rhos[k] = spearmanr(x[idx], y[idx]).statistic
    return float(np.percentile(rhos, 2.5)), float(np.percentile(rhos, 97.5))

conn = pairs.loc[pairs.connected].copy()
# Spearman is rank-invariant, so no transform of `syn_size_sum` is needed for the
# primary statistic. log10(...) is computed only for the Pearson sensitivity check
# and for plotting, since `syn_size_sum` is heavy-tailed (max ~ 6e5).
conn['log10_size_sum'] = np.log10(conn['syn_size_sum'].clip(lower=1.0))
conn['log10_size_max'] = np.log10(conn['syn_size_max'].clip(lower=1.0))

rho_h2,    p_rho_h2    = spearmanr(conn['syn_size_sum'], conn['f_corr'])
rho_max,   p_rho_max   = spearmanr(conn['syn_size_max'], conn['f_corr'])
r_pearson, p_pearson   = pearsonr(conn['log10_size_sum'], conn['f_corr'])
ci_lo_h2,  ci_hi_h2    = boot_spearman_ci(conn['syn_size_sum'].to_numpy(),
                                          conn['f_corr'].to_numpy())
r2_log = float(r_pearson ** 2)

print(f'H2 - connected pairs only (n={len(conn):,})')
print(f'  Spearman rho (syn_size_sum, f_corr)   = {rho_h2:+.4f}    '
      f'95% boot CI [{ci_lo_h2:+.4f}, {ci_hi_h2:+.4f}]    p = {p_rho_h2:.2e}    (primary, two-sided)')
print(f'  Spearman rho (syn_size_max, f_corr)   = {rho_max:+.4f}    p = {p_rho_max:.2e}    (sensitivity check)')
print(f'  Pearson  r   (log10 size_sum, f_corr) = {r_pearson:+.4f}    p = {p_pearson:.2e}')
print(f'  Pearson  r^2 on log10 strength        = {r2_log:.4f}    '
      f'(<1% of F_corr variance is *linearly* explained by log strength)')
print('  NOTE: Spearman rho^2 is NOT a fraction of variance explained;')
print('        we report the Pearson r^2 on log10 strength as the proper variance-explained scale.')

conn['quartile'] = pd.qcut(conn['syn_size_sum'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
q = conn.groupby('quartile', observed=True)['f_corr'].agg(['count', 'mean', 'sem'])
print()
print('quartile means of F_corr (Q1 = weakest, Q4 = strongest):')
print(q)

H2_RESULT = {
    'hypothesis':         'H2',
    'description':        'syn_size_sum vs F_corr (connected only)',
    'test_direction':     'two-sided Spearman',
    'n_connected':        int(len(conn)),
    'spearman_rho_sum':   float(rho_h2),
    'spearman_ci_lo':     float(ci_lo_h2),
    'spearman_ci_hi':     float(ci_hi_h2),
    'p_spearman_sum':     float(p_rho_h2),
    'spearman_rho_max':   float(rho_max),
    'p_spearman_max':     float(p_rho_max),
    'pearson_log10_r':    float(r_pearson),
    'pearson_log10_r2':   r2_log,
    'p_pearson_log10':    float(p_pearson),
}

In [ ]:
# ---- neuron-identity permutation null for H2 ----
# For each permutation pi:
#   - syn_ij_perm = C[pi[i], pi[j]],  syn_ji_perm = C[pi[j], pi[i]]
#   - syn_sum_perm = syn_ij_perm + syn_ji_perm
#   - 'connected' pairs are those with syn_sum_perm > 0
#   - Spearman rho is recomputed on (syn_sum_perm, f_corr) over those pairs.
# F_corr and C are both fixed; only neuron labels move. Two-sided test.

N_PERM_H2 = 5000
rng = np.random.default_rng(RNG_SEED + 1)
obs_rho_h2 = float(rho_h2)
perm_rhos_h2 = np.empty(N_PERM_H2)
for k in range(N_PERM_H2):
    pi = rng.permutation(N)
    s_ij = C_dense[pi[iu_all], pi[ju_all]]
    s_ji = C_dense[pi[ju_all], pi[iu_all]]
    s_sum = s_ij + s_ji
    conn_k = s_sum > 0
    if conn_k.sum() < 5:
        perm_rhos_h2[k] = 0.0
        continue
    perm_rhos_h2[k] = spearmanr(s_sum[conn_k], f_pairs_arr[conn_k]).statistic

n_exceed_h2 = int((np.abs(perm_rhos_h2) >= abs(obs_rho_h2)).sum())
p_perm_h2 = (n_exceed_h2 + 1) / (N_PERM_H2 + 1)
null_mu_h2 = float(perm_rhos_h2.mean())
null_sd_h2 = float(perm_rhos_h2.std(ddof=1))
z_obs_h2 = (obs_rho_h2 - null_mu_h2) / null_sd_h2 if null_sd_h2 > 0 else float('inf')

print(f'H2 neuron-identity permutation null (n_perm = {N_PERM_H2:,})')
print(f'  observed Spearman rho   = {obs_rho_h2:+.4f}')
print(f'  null mean +/- std       = {null_mu_h2:+.4f} +/- {null_sd_h2:.4f}')
print(f'  observed sits at z      = {z_obs_h2:+.2f}  std deviations from null mean')
print(f'  two-sided perm p-value  = {p_perm_h2:.4e}   (#perms |rho| >= |obs|: {n_exceed_h2}/{N_PERM_H2})')

H2_RESULT['perm_n']           = int(N_PERM_H2)
H2_RESULT['perm_null_mean']   = null_mu_h2
H2_RESULT['perm_null_std']    = null_sd_h2
H2_RESULT['perm_z']           = float(z_obs_h2)
H2_RESULT['perm_p_two_sided'] = float(p_perm_h2)

### H2 Conclusion

H2 is supported, but weakly. Stronger connected pairs tend to have slightly higher signal correlation, so the structural edge weight contains some additional information beyond the binary connected label. However, the effect is small and explains very little of the variance in pairwise functional similarity.

The practical conclusion is that summed synapse size should not replace H1 as the main result. It refines H1 by showing a weak graded trend among connected pairs, but the dominant observation remains the binary connected-versus-unconnected shift. In the report, this should be framed as evidence that connection strength nudges expected functional similarity upward rather than as evidence for a strong quantitative prediction from synapse size.


## 6. H3. Reciprocity

H3 asks whether bidirectional connectivity adds functional similarity beyond the presence of a one-way connection. This matters because the structural graph is directed, but the functional correlation matrix is symmetric. A bidirectional pair is one biologically interpretable way that directed anatomy might map onto symmetric functional similarity.

The analysis compares three groups: unconnected pairs, unidirectionally connected pairs, and bidirectionally connected pairs. The key caution is sample size: bidirectional pairs are rare in this cohort, so the section must separate the observed ordering of group means from what is statistically resolved.


In [ ]:
def boot_mean_ci(x, n_boot=4000, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    bm = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(n_boot)])
    return float(x.mean()), float(np.percentile(bm, 2.5)), float(np.percentile(bm, 97.5))

groups = {'none': f_none, 'uni': f_uni, 'bi': f_bi}
group_stats = {}
for g in ORDER_CONN:
    m_, lo_, hi_ = boot_mean_ci(groups[g])
    group_stats[g] = {'n': int(len(groups[g])), 'mean': m_, 'ci_lo': lo_, 'ci_hi': hi_}
    print(f'{g:>4s}  n={len(groups[g]):>7,d}   mean f_corr = {m_:+.4f}   95% CI [{lo_:+.4f}, {hi_:+.4f}]')

pairwise = []
for name, a, b in [('bi > none', f_bi, f_none),
                   ('bi > uni',  f_bi, f_uni),
                   ('uni > none', f_uni, f_none)]:
    U_, p_ = mannwhitneyu(a, b, alternative='greater')
    pairwise.append((name, U_, p_))
    print(f'  {name:>11s}   U = {U_:.3e}   one-sided p = {p_:.3e}')

# ---- Power-aware framing of the bi > uni non-significance ----
# Cohen's d for the observed bi-uni gap, and for the uni-none gap as a reference.
def _pooled_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return float((a.mean() - b.mean()) / sp)

d_uni_none = _pooled_d(f_uni, f_none)
d_bi_uni   = _pooled_d(f_bi,  f_uni)
print()
print(f"  Cohen's d (uni - none) = {d_uni_none:+.3f}    (the 'uni > none' effect)")
print(f"  Cohen's d (bi  - uni)  = {d_bi_uni:+.3f}    (the actual 'bi > uni' gap; small)")
print( '  Reading: with n_bi = 242, the design has good power for d = 0.24,')
print( '          but only modest power for d ~ 0.09. The data therefore *exclude*')
print( '          a bi - uni gap as large as the uni - none step, while not ruling')
print( '          out a small positive bi - uni effect.')

# ---- Area composition of bi vs uni pairs (is the near-tie geographic?) ----
ba = matched['brain_area'].astype(str).to_numpy()
def area_breakdown(idx_pairs):
    a = ba[idx_pairs[:, 0]]; b = ba[idx_pairs[:, 1]]
    same = (a == b)
    labels = np.where(same, np.char.add(np.char.add(a, '-'), b), 'cross-area')
    return pd.Series(labels).value_counts().to_dict()

bi_ij  = pairs.loc[pairs.conn_type == 'bi',  ['i', 'j']].to_numpy()
uni_ij = pairs.loc[pairs.conn_type == 'uni', ['i', 'j']].to_numpy()
print()
print(f'  bi  pairs (n={len(bi_ij):>5,d})  by area : {area_breakdown(bi_ij)}')
print(f'  uni pairs (n={len(uni_ij):>5,d})  by area : {area_breakdown(uni_ij)}')

H3_RESULT = {
    'hypothesis':            'H3',
    'description':           'reciprocity premium (bi > uni > none)',
    'test_direction':        'one-sided pairwise (a > b for each comparison)',
    'n_none':                group_stats['none']['n'],
    'n_uni':                 group_stats['uni']['n'],
    'n_bi':                  group_stats['bi']['n'],
    'mean_none':             group_stats['none']['mean'],
    'mean_uni':              group_stats['uni']['mean'],
    'mean_bi':               group_stats['bi']['mean'],
    'ci_lo_bi':              group_stats['bi']['ci_lo'],
    'ci_hi_bi':              group_stats['bi']['ci_hi'],
    'cohens_d_uni_vs_none':  d_uni_none,
    'cohens_d_bi_vs_uni':    d_bi_uni,
    'p_uni_gt_none':         float(pairwise[2][2]),
    'p_bi_gt_none':          float(pairwise[0][2]),
    'p_bi_gt_uni':           float(pairwise[1][2]),
}

In [ ]:
# ---- (a) V1-V1-stratified pairwise tests ----
# Restricting to V1-V1 pairs removes the bi-vs-uni area-composition difference
# shown in section 7 (bi pairs are 96.3% V1-V1; uni pairs are 90.0% V1-V1).
i_idx = pairs['i'].to_numpy()
j_idx = pairs['j'].to_numpy()
v1v1_mask = (ba[i_idx] == 'V1') & (ba[j_idx] == 'V1')
pairs_v1 = pairs.loc[v1v1_mask].copy()

f_none_v1 = pairs_v1.loc[pairs_v1.conn_type == 'none', 'f_corr'].to_numpy()
f_uni_v1  = pairs_v1.loc[pairs_v1.conn_type == 'uni',  'f_corr'].to_numpy()
f_bi_v1   = pairs_v1.loc[pairs_v1.conn_type == 'bi',   'f_corr'].to_numpy()

print('V1-V1-only pair counts        :', {
    'none': int(len(f_none_v1)),
    'uni':  int(len(f_uni_v1)),
    'bi':   int(len(f_bi_v1)),
})
print('V1-V1-only mean f_corr        :', {
    'none': float(f_none_v1.mean()),
    'uni':  float(f_uni_v1.mean()),
    'bi':   float(f_bi_v1.mean()),
})

U_uni_none_v1, p_uni_none_v1 = mannwhitneyu(f_uni_v1, f_none_v1, alternative='greater')
U_bi_none_v1,  p_bi_none_v1  = mannwhitneyu(f_bi_v1,  f_none_v1, alternative='greater')
U_bi_uni_v1,   p_bi_uni_v1   = mannwhitneyu(f_bi_v1,  f_uni_v1,  alternative='greater')

d_uni_none_v1 = _pooled_d(f_uni_v1, f_none_v1)
d_bi_uni_v1   = _pooled_d(f_bi_v1,  f_uni_v1)

print(f'  V1-V1 uni > none : U = {U_uni_none_v1:.3e}  p = {p_uni_none_v1:.3e}  d = {d_uni_none_v1:+.3f}')
print(f'  V1-V1 bi  > none : U = {U_bi_none_v1:.3e}  p = {p_bi_none_v1:.3e}')
print(f'  V1-V1 bi  > uni  : U = {U_bi_uni_v1:.3e}  p = {p_bi_uni_v1:.3e}  d = {d_bi_uni_v1:+.3f}')
print('  Reading: removing the V1-vs-RL/AL composition difference between bi and uni,')
print('           the bi - uni gap can be re-evaluated on a like-for-like sample.')

# ---- (b) neuron-identity permutation null for the three H3 mean-differences ----
# Same null structure as section 5b, but with three test statistics:
#   uni - none, bi - none, bi - uni
# Each pair's conn_type is recomputed from the permuted directed adjacency, and
# the three group means are recomputed against the original f_corr values.
A_pre_dense = (C_dense > 0)

N_PERM_H3 = 5000
rng = np.random.default_rng(RNG_SEED + 2)
obs_d_uni_none = float(f_uni.mean() - f_none.mean())
obs_d_bi_uni   = float(f_bi.mean()  - f_uni.mean())
obs_d_bi_none  = float(f_bi.mean()  - f_none.mean())

perm_d_uni_none = np.empty(N_PERM_H3)
perm_d_bi_uni   = np.empty(N_PERM_H3)
perm_d_bi_none  = np.empty(N_PERM_H3)

for k in range(N_PERM_H3):
    pi = rng.permutation(N)
    has_ij = A_pre_dense[pi[iu_all], pi[ju_all]]
    has_ji = A_pre_dense[pi[ju_all], pi[iu_all]]
    is_bi   = has_ij & has_ji
    is_uni  = has_ij ^ has_ji
    is_none = ~(has_ij | has_ji)
    m_none = f_pairs_arr[is_none].mean() if is_none.any() else 0.0
    m_uni  = f_pairs_arr[is_uni].mean()  if is_uni.any()  else 0.0
    m_bi   = f_pairs_arr[is_bi].mean()   if is_bi.any()   else 0.0
    perm_d_uni_none[k] = m_uni - m_none
    perm_d_bi_uni[k]   = m_bi  - m_uni
    perm_d_bi_none[k]  = m_bi  - m_none

def _perm_p_z(perm, obs):
    n_ex = int((perm >= obs).sum())
    p = (n_ex + 1) / (len(perm) + 1)
    mu = float(perm.mean())
    sd = float(perm.std(ddof=1))
    z = (obs - mu) / sd if sd > 0 else float('inf')
    return p, mu, sd, z, n_ex

p_perm_uni_none, mu_un, sd_un, z_un, _ = _perm_p_z(perm_d_uni_none, obs_d_uni_none)
p_perm_bi_uni,   mu_bu, sd_bu, z_bu, _ = _perm_p_z(perm_d_bi_uni,   obs_d_bi_uni)
p_perm_bi_none,  mu_bn, sd_bn, z_bn, _ = _perm_p_z(perm_d_bi_none,  obs_d_bi_none)

print()
print(f'H3 neuron-identity permutation null (n_perm = {N_PERM_H3:,}, all one-sided a > b)')
print(f'  uni - none : observed = {obs_d_uni_none:+.4f}  '
      f'null = {mu_un:+.4f} +/- {sd_un:.4f}  z = {z_un:+.2f}  p = {p_perm_uni_none:.4e}')
print(f'  bi  - none : observed = {obs_d_bi_none:+.4f}  '
      f'null = {mu_bn:+.4f} +/- {sd_bn:.4f}  z = {z_bn:+.2f}  p = {p_perm_bi_none:.4e}')
print(f'  bi  - uni  : observed = {obs_d_bi_uni:+.4f}  '
      f'null = {mu_bu:+.4f} +/- {sd_bu:.4f}  z = {z_bu:+.2f}  p = {p_perm_bi_uni:.4e}')

H3_RESULT['n_v1v1_none']               = int(len(f_none_v1))
H3_RESULT['n_v1v1_uni']                = int(len(f_uni_v1))
H3_RESULT['n_v1v1_bi']                 = int(len(f_bi_v1))
H3_RESULT['mean_v1v1_none']            = float(f_none_v1.mean())
H3_RESULT['mean_v1v1_uni']             = float(f_uni_v1.mean())
H3_RESULT['mean_v1v1_bi']              = float(f_bi_v1.mean())
H3_RESULT['p_v1v1_uni_gt_none']        = float(p_uni_none_v1)
H3_RESULT['p_v1v1_bi_gt_none']         = float(p_bi_none_v1)
H3_RESULT['p_v1v1_bi_gt_uni']          = float(p_bi_uni_v1)
H3_RESULT['cohens_d_v1v1_uni_vs_none'] = float(d_uni_none_v1)
H3_RESULT['cohens_d_v1v1_bi_vs_uni']   = float(d_bi_uni_v1)
H3_RESULT['perm_n']                    = int(N_PERM_H3)
H3_RESULT['perm_p_uni_gt_none']        = float(p_perm_uni_none)
H3_RESULT['perm_p_bi_gt_none']         = float(p_perm_bi_none)
H3_RESULT['perm_p_bi_gt_uni']          = float(p_perm_bi_uni)
H3_RESULT['perm_z_uni_gt_none']        = float(z_un)
H3_RESULT['perm_z_bi_gt_none']         = float(z_bn)
H3_RESULT['perm_z_bi_gt_uni']          = float(z_bu)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# --- (A) H1: violin of f_corr by conn_type ---
ax = axes[0, 0]
data = [f_none, f_uni, f_bi]
parts = ax.violinplot(data, showmeans=False, showmedians=True, widths=0.85)
for k, body in enumerate(parts['bodies']):
    body.set_facecolor(PAL_CONN[ORDER_CONN[k]])
    body.set_edgecolor('white')
    body.set_alpha(0.85)
for key in ('cmedians', 'cbars', 'cmins', 'cmaxes'):
    parts[key].set_color('0.25')
    parts[key].set_linewidth(1.0)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels([f'none\nn={len(f_none):,}',
                    f'uni\nn={len(f_uni):,}',
                    f'bi\nn={len(f_bi):,}'])
ax.set_ylabel(r'Pearson signal correlation $F_{corr}$')
ax.axhline(0, color='0.4', lw=0.7, ls=':')
ax.set_title(f'A . H1 - $\\Delta$mean (conn-none) = {md_h1:+.4f},  d = {d_h1:+.3f} (small)\n'
             f'MW one-sided p = {p_h1:.1e};  perm p (n={N_PERM:,}) = {p_perm:.1e},  z = {z_obs:+.1f}$\\sigma$')

# --- (B) H2 scatter: hexbin of log10 syn_size_sum vs f_corr (with binned-median overlay) ---
ax = axes[0, 1]
hb = ax.hexbin(conn['log10_size_sum'], conn['f_corr'],
               gridsize=40, mincnt=1, cmap='viridis', bins='log')
xb = np.linspace(conn['log10_size_sum'].min(), conn['log10_size_sum'].max(), 13)
ix = np.digitize(conn['log10_size_sum'].to_numpy(), xb)
xc, yb = [], []
for k in range(1, len(xb)):
    sel = (ix == k)
    if sel.sum() >= 20:
        xc.append(0.5 * (xb[k-1] + xb[k]))
        yb.append(np.median(conn['f_corr'].to_numpy()[sel]))
ax.plot(xc, yb, 'o-', color='white', mec='black', mfc='white', lw=1.5, ms=5,
        label='binned median')
ax.axhline(0, color='0.4', lw=0.7, ls=':')
ax.set_xlabel(r'$\log_{10}$ summed synapse size (cleft volume)')
ax.set_ylabel(r'$F_{corr}$')
ax.set_title(f'B . H2 scatter - Spearman $\\rho$(sum) = {rho_h2:+.3f}  '
             f'95% boot CI [{ci_lo_h2:+.3f}, {ci_hi_h2:+.3f}]\n'
             f'(n = {len(conn):,};  perm p = {p_perm_h2:.1e}, z = {z_obs_h2:+.1f}$\\sigma$;  '
             f'Pearson $r^2$ on log strength = {r2_log:.3f})')
ax.legend(loc='upper left', frameon=False, fontsize=8)
cb = plt.colorbar(hb, ax=ax, fraction=0.045, pad=0.02)
cb.set_label('pair count (log)')

# --- (C) H2 quartile means of f_corr ---
ax = axes[1, 0]
qx = np.arange(4)
ax.bar(qx, q['mean'], yerr=q['sem'],
       color=plt.cm.viridis(np.linspace(0.25, 0.85, 4)),
       edgecolor='white', linewidth=0.8, capsize=4)
for k, (mu, n_) in enumerate(zip(q['mean'], q['count'])):
    ax.text(k, mu + 0.005, f'n={n_}', ha='center', va='bottom', fontsize=9, color='0.25')
ax.set_xticks(qx); ax.set_xticklabels(q.index)
ax.set_ylabel(r'mean $F_{corr}$  ($\pm$ SEM)')
ax.set_xlabel('synapse-strength quartile (Q1 = weakest)')
ax.set_title(f'C . H2 quartile means - roughly monotone, weak gradient\n'
             f'sensitivity: $\\rho$(syn_size_max) = {rho_max:+.3f}, p = {p_rho_max:.1e}')

# --- (D) H3: mean +/- 95% CI by conn_type ---
ax = axes[1, 1]
xs = np.arange(3)
mu_  = [group_stats[g]['mean']  for g in ORDER_CONN]
los_ = [group_stats[g]['ci_lo'] for g in ORDER_CONN]
his_ = [group_stats[g]['ci_hi'] for g in ORDER_CONN]
err_lo = [m - lo for m, lo in zip(mu_, los_)]
err_hi = [hi - m for m, hi in zip(mu_, his_)]
ax.bar(xs, mu_, yerr=[err_lo, err_hi],
       color=[PAL_CONN[g] for g in ORDER_CONN],
       edgecolor='white', linewidth=0.8, capsize=6)
for k, g in enumerate(ORDER_CONN):
    ax.text(k, mu_[k] + 0.005, f'n={group_stats[g]["n"]:,}',
            ha='center', va='bottom', fontsize=9, color='0.25')
ax.set_xticks(xs); ax.set_xticklabels(ORDER_CONN)
ax.set_ylabel(r'mean $F_{corr}$  (95% bootstrap CI)')
ax.set_title(
    f'D . H3 - bi>uni MW p: all-area = {H3_RESULT["p_bi_gt_uni"]:.2f},  '
    f'V1-V1 only = {H3_RESULT["p_v1v1_bi_gt_uni"]:.2f}  '
    f'(perm p = {H3_RESULT["perm_p_bi_gt_uni"]:.2f})\n'
    f'd(uni-none) = {d_uni_none:+.2f},  d(bi-uni) = {d_bi_uni:+.2f}  '
    f'(power-limited, not size-limited)'
)

fig.suptitle('Person 1 (Kevin) - H1-H3 pair-level baseline . 906 cohort, signal correlation',
             y=1.00, fontsize=11)
fig.tight_layout()

fig_path = FIG_DIR / 'person1_h1_h3_pair_baseline.png'
fig.savefig(fig_path, bbox_inches='tight')
plt.show()
print(f'figure saved to: {fig_path}')

summary = pd.DataFrame([H1_RESULT, H2_RESULT, H3_RESULT])
tbl_path = TBL_DIR / 'person1_h1_h3_summary.csv'
summary.to_csv(tbl_path, index=False)
print(f'table saved to : {tbl_path}')
print()
print('summary table (one row per hypothesis; NaN = column not applicable to that test):')
with pd.option_context('display.max_columns', None, 'display.width', 220):
    print(summary.to_string(index=False))
summary

### H3 Conclusion

H3 is only partially supported. The mean ordering is in the expected direction, with bidirectional pairs highest, unidirectional pairs intermediate, and unconnected pairs lowest. The robust part of the result is that connected pairs, whether unidirectional or bidirectional, are above unconnected pairs.

The specific reciprocity premium is not resolved: the bidirectional-versus-unidirectional difference is small and not significant under the current tests. This means reciprocity should not be used as a headline claim. A conservative interpretation is that bidirectionality may carry a small positive signal, but this cohort does not provide strong evidence that reciprocal pairs are meaningfully more functionally similar than one-way connected pairs after connectedness itself is accounted for.


## 7. H4-H6. Controls for Distance, Composition, and Orientation

H4-H6 are the credibility layer for H1. A connected-pair effect is interesting only if it is not just a byproduct of neurons being close together, belonging to the same anatomical or cell-type groups, or sharing basic visual tuning.

These controls should not be read as independent discoveries on the same level as H1. Their job is to test alternative explanations. If the H1 effect shrinks but remains positive after these controls, the correct conclusion is not that confounds are irrelevant; it is that direct connectivity still carries functional information after accounting for major known structure in the data.


In [ ]:
# 0.5  Tiny shared helpers reused by H4, H5, H6
def boot_mean_diff(a, b, n_boot=4000, seed=0):
    """Bootstrap (mean(a) - mean(b)) with a 95% percentile interval."""
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        diffs[k] = rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
    return float(diffs.mean()), tuple(np.percentile(diffs, [2.5, 97.5]))

def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std(ddof=0)

# Pre-split f_conn / f_un — used by every H4 panel
f_conn = pairs.loc[ pairs.connected, 'f_corr'].to_numpy()
f_un   = pairs.loc[~pairs.connected, 'f_corr'].to_numpy()

# Container for the final cross-hypothesis summary table
RESULTS = []

## 7.1 H4. Distance Control

Distance is the most important confound for the pair-level result. Nearby neurons are more likely to be connected, and nearby neurons may also have more similar functional responses because of shared cortical location, shared inputs, or local organization. H4 therefore asks whether H1 survives after distance is taken seriously.

The section uses three complementary checks. The distance-profile plot shows why distance matters. The distance-matched retest compares connected pairs to unconnected pairs at similar soma-soma distances. The within-bin and logistic analyses ask whether the connected-pair effect remains positive when distance is handled more systematically.


In [ ]:
# H4 (a) · bin pairs by soma-soma distance and read off the two curves
h4a_edges = np.linspace(pairs.dist_um.min(), pairs.dist_um.max(), 25)
h4a_mids  = 0.5 * (h4a_edges[:-1] + h4a_edges[1:])
h4a_bin   = pairs.groupby(pd.cut(pairs.dist_um, h4a_edges, include_lowest=True), observed=False)
h4a_pconn = h4a_bin.connected.mean().to_numpy()
h4a_fc    = h4a_bin.f_corr.mean().to_numpy()
h4a_n     = h4a_bin.size().to_numpy()

# Pearson correlation between P(conn) and mean F_corr across bins gives a one-line summary
from scipy.stats import pearsonr
r_pf, p_pf = pearsonr(h4a_pconn, h4a_fc)
print(f'across {len(h4a_mids)} distance bins, corr( P(conn), mean F_corr ) = {r_pf:+.3f}  (p = {p_pf:.2e})')
print(f'span: {h4a_edges[0]:.0f} – {h4a_edges[-1]:.0f} µm,   first-bin pairs n={h4a_n[0]:,}')

In [ ]:
# H4 (a) · twin-axis decay plot
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax2 = ax.twinx()
ax.plot(h4a_mids,  h4a_pconn, lw=2.2, color=PAL_CONN['uni'], label='P(connected)')
ax2.plot(h4a_mids, h4a_fc,    lw=2.2, color=PAL_CONN['bi'], ls='--', label=r'mean $F_{corr}$')
ax.set_xlabel('soma–soma distance  (µm)')
ax.set_ylabel('P(connected)',          color=PAL_CONN['uni'])
ax2.set_ylabel(r'mean $F_{corr}$',     color=PAL_CONN['bi'])
ax.tick_params (axis='y', colors=PAL_CONN['uni'])
ax2.tick_params(axis='y', colors=PAL_CONN['bi'])
ax.grid(False); ax2.grid(False)
ax.set_title('H4(a) · Both quantities decay with distance')
lg = [Line2D([0], [0], color=PAL_CONN['uni'], lw=2.2,             label='P(connected)'),
      Line2D([0], [0], color=PAL_CONN['bi'],  lw=2.2, ls='--',    label=r'mean $F_{corr}$')]
ax.legend(handles=lg, loc='upper right',
          title=fr'corr across bins: r = {r_pf:+.2f}', title_fontsize=8.5)
fig.tight_layout()
fig.savefig(OUT_FIG / 'H4a_distance_profile.png')
plt.show()

In [ ]:
# H4 (b) · for each connected pair pick the closest-distance unconnected pair, no replacement
rng        = np.random.default_rng(0)
conn_idx   = pairs.index[ pairs.connected].to_numpy()
un_idx     = pairs.index[~pairs.connected].to_numpy()
un_sorted  = un_idx[np.argsort(pairs.loc[un_idx, 'dist_um'].to_numpy())]
un_d       = pairs.loc[un_sorted, 'dist_um'].to_numpy()
used       = np.zeros(len(un_sorted), dtype=bool)
matched_un = []
for ci in rng.permutation(conn_idx):
    d   = pairs.loc[ci, 'dist_um']
    pos = np.searchsorted(un_d, d)
    best, best_d = None, np.inf
    for off in range(0, 50):
        for k in (pos - off, pos + off):
            if 0 <= k < len(un_sorted) and not used[k]:
                diff = abs(un_d[k] - d)
                if diff < best_d:
                    best, best_d = k, diff
        if best is not None and best_d < 2.0:
            break
    if best is not None:
        used[best] = True
        matched_un.append(un_sorted[best])
matched_un = np.asarray(matched_un)
f_match    = pairs.loc[matched_un, 'f_corr'].to_numpy()

_, p_match         = mannwhitneyu(f_conn, f_match, alternative='greater')
md_match, ci_match = boot_mean_diff(f_conn, f_match, seed=1)
md_raw,   ci_raw   = boot_mean_diff(f_conn, f_un,    seed=2)
print(f'raw            Δmean = {md_raw:+.4f}   CI [{ci_raw[0]:+.4f}, {ci_raw[1]:+.4f}]')
print(f'distance-matched Δmean = {md_match:+.4f}   CI [{ci_match[0]:+.4f}, {ci_match[1]:+.4f}]   p = {p_match:.2e}')
print(f'matched control n   = {len(matched_un):,}  /  connected n = {len(conn_idx):,}')

RESULTS += [
    {'hypothesis': 'H4', 'test': 'raw H1 (no control)',
     'mean_diff': md_raw,   'ci_lo': ci_raw[0],   'ci_hi': ci_raw[1],   'p': float('nan'),
     'note': 'baseline, included for shrinkage comparison'},
    {'hypothesis': 'H4', 'test': 'distance-matched',
     'mean_diff': md_match, 'ci_lo': ci_match[0], 'ci_hi': ci_match[1], 'p': p_match,
     'note': f'n_matched={len(matched_un)}'},
]

In [ ]:
# H4 (b) · histograms of f_corr for three groups
fig, ax = plt.subplots(figsize=(5.6, 3.6))
for label, x, c in [
    ('unconnected (all)',          f_un,    PAL_CONN['none']),
    ('unconnected (dist-matched)', f_match, '#7F9BAE'),
    ('connected',                  f_conn,  PAL_CONN['uni']),
]:
    ax.hist(x, bins=80, density=True, histtype='step', linewidth=1.8,
            color=c, label=f'{label} (n={len(x):,})')
ax.axvline(0, color='0.4', lw=0.6, ls=':')
ax.set_xlabel(r'$F_{corr}$')
ax.set_ylabel('density')
ax.set_xlim(-0.5, 1.0)
ax.set_title('H4(b) · Distance-matched comparison')
ax.text(0.02, 0.97,
        f'Δmean = {md_match:+.3f}\nCI [{ci_match[0]:+.3f}, {ci_match[1]:+.3f}]\np = {p_match:.1e}',
        transform=ax.transAxes, va='top', ha='left', fontsize=8.6, color='0.25',
        bbox=dict(boxstyle='round,pad=0.32', facecolor='white', edgecolor='0.75', lw=0.7))
ax.legend(loc='upper right')
fig.tight_layout()
fig.savefig(OUT_FIG / 'H4b_distance_matched.png')
plt.show()

In [ ]:
# H4 (c) · split pairs into five distance quintiles, retest H1 inside each, BH-FDR correct
qs = pairs.dist_um.quantile(np.linspace(0, 1, 6)).to_numpy()
labels_q, ds_q, lo_q, hi_q, ps_q, n_c, n_u = [], [], [], [], [], [], []
for k in range(5):
    sel = (pairs.dist_um >= qs[k]) & ((pairs.dist_um < qs[k + 1]) if k < 4 else (pairs.dist_um <= qs[k + 1]))
    sub = pairs[sel]
    a = sub.loc[ sub.connected, 'f_corr'].to_numpy()
    b = sub.loc[~sub.connected, 'f_corr'].to_numpy()
    if len(a) < 5 or len(b) < 5:
        continue
    md_, ci_ = boot_mean_diff(a, b, n_boot=2000, seed=k + 10)
    _, p_    = mannwhitneyu(a, b, alternative='greater')
    labels_q.append(f'Q{k+1}: {qs[k]:.0f}\u2013{qs[k+1]:.0f} µm')
    ds_q.append(md_); lo_q.append(ci_[0]); hi_q.append(ci_[1])
    ps_q.append(p_); n_c.append(len(a)); n_u.append(len(b))
ranked = np.argsort(ps_q)
m      = len(ps_q)
q_adj  = np.empty(m); cur = 1.0
for r in range(m - 1, -1, -1):
    idx = ranked[r]; cur = min(cur, ps_q[idx] * m / (r + 1)); q_adj[idx] = cur

h4c = pd.DataFrame({
    'bin': labels_q, 'n_conn': n_c, 'n_unconn': n_u,
    'mean_diff': ds_q, 'ci_lo': lo_q, 'ci_hi': hi_q, 'p': ps_q, 'q': q_adj,
})
print(h4c.to_string(index=False))
for r in h4c.itertuples():
    RESULTS.append({'hypothesis': 'H4', 'test': f'within-bin {r.bin.split(":")[0]}',
                    'mean_diff': r.mean_diff, 'ci_lo': r.ci_lo, 'ci_hi': r.ci_hi,
                    'p': r.p, 'note': f'BH q={r.q:.1e}; n_conn={r.n_conn}, n_unconn={r.n_unconn}'})

In [ ]:
# H4 (c) · forest plot of within-bin Δ mean F_corr
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ypos = np.arange(len(h4c))[::-1]
cols = RAMP_SEQ(np.linspace(0.25, 0.85, len(h4c)))
for y_, r, c_ in zip(ypos, h4c.itertuples(), cols):
    ax.errorbar(r.mean_diff, y_, xerr=[[r.mean_diff - r.ci_lo], [r.ci_hi - r.mean_diff]],
                fmt='o', color=c_, markersize=8, capsize=4, lw=1.6, mec='white', mew=0.7)
    ax.text(r.ci_hi + 0.0015, y_, f'q={r.q:.1e}', va='center', fontsize=8.4, color='0.3')
ax.axvline(0, color='0.4', lw=0.7, ls=':')
ax.set_yticks(ypos); ax.set_yticklabels(h4c.bin)
ax.set_xlabel(r'$\Delta$ mean $F_{corr}$  (connected − unconnected, 95% CI)')
ax.set_title('H4(c) · Within-distance-bin tests (BH-FDR)')
ax.margins(x=0.20)
fig.tight_layout()
fig.savefig(OUT_FIG / 'H4c_within_bin.png')
plt.show()

In [ ]:
# H4 (d) · partial logistic regression, connected ~ f_corr + dist + dist²  (HC0 SEs)
Xh4 = np.column_stack([pairs.f_corr.values, pairs.dist_um.values, pairs.dist_um.values ** 2])
Xh4 = sm.add_constant(Xh4)
yh4 = pairs.connected.astype(int).to_numpy()
logit_h4 = sm.Logit(yh4, Xh4).fit(disp=False, cov_type='HC0')
beta_fc, se_fc, p_fc = logit_h4.params[1], logit_h4.bse[1], logit_h4.pvalues[1]
print(f'logistic β(f_corr | dist, dist²) = {beta_fc:+.3f} ± {se_fc:.3f}   p = {p_fc:.2e}')

RESULTS += [{
    'hypothesis': 'H4', 'test': 'logistic β(f_corr | dist, dist²)',
    'mean_diff': beta_fc,  'ci_lo': beta_fc - 1.96 * se_fc, 'ci_hi': beta_fc + 1.96 * se_fc, 'p': p_fc,
    'note': 'connected ~ f_corr + dist + dist², HC0 SE',
}]

In [ ]:
# H4 (d) · two H1 estimates (raw vs distance-matched) + the partial logistic readout
fig, ax = plt.subplots(figsize=(5.4, 3.7))
estimates = [
    ('H1 raw\n(no control)',  md_raw,   ci_raw[0],   ci_raw[1],   PAL_CONN['none']),
    ('distance-\nmatched',    md_match, ci_match[0], ci_match[1], '#7F9BAE'),
]
xpos = np.arange(len(estimates))
for x_, (lbl, m_, lo_, hi_, c_) in zip(xpos, estimates):
    ax.errorbar(x_, m_, yerr=[[m_ - lo_], [hi_ - m_]], fmt='o',
                color=c_, markersize=11, capsize=5, lw=2.0, mec='white', mew=0.8)

# arrow showing the shrinkage between raw and matched, plus a clear annotation
ax.annotate('', xy=(xpos[1], md_match), xytext=(xpos[0], md_raw),
            arrowprops=dict(arrowstyle='->', color='0.55', lw=1.2,
                            connectionstyle='arc3,rad=-0.18'))

# Δ labels — placed to the side of each marker so they don't crash into the cap
ax.annotate(f'Δ = {md_raw:+.3f}',   xy=(xpos[0], md_raw),   xytext=(8, 0),
            textcoords='offset points', va='center', ha='left',
            fontsize=9.2, color='0.25')
ax.annotate(f'Δ = {md_match:+.3f}', xy=(xpos[1], md_match), xytext=(-8, 0),
            textcoords='offset points', va='center', ha='right',
            fontsize=9.2, color='0.25')

# shrinkage label just above the curve apex
ax.text(0.5 * (xpos[0] + xpos[1]),
        max(md_raw, md_match) + 0.0042,
        f'shrinkage  −{(md_raw - md_match) / md_raw:.0%}',
        ha='center', va='bottom', fontsize=9.0, color='0.45',
        fontstyle='italic')

ax.axhline(0, color='0.4', lw=0.7, ls=':')
ax.set_xticks(xpos); ax.set_xticklabels([e[0] for e in estimates], fontsize=9.5)
ax.set_xlim(-0.55, 1.55)
ax.set_ylabel(r'$\Delta$ mean $F_{corr}$  (95% CI)')
ax.set_ylim(-0.001, max(ci_raw[1], ci_match[1]) * 1.65)
ax.set_title('H4(d) · Shrinkage of H1 under distance control')

# Bottom-centre is empty (curve dips toward distance-matched on the right) — anchor box there.
ax.text(0.50, 0.04,
        f'partial logistic\nβ($F_{{corr}}$ | dist, dist²) = {beta_fc:+.2f} ± {1.96 * se_fc:.2f}    p = {p_fc:.1e}',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=8.8, color='0.25',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#F5F5F5', edgecolor='0.7', lw=0.8))
fig.tight_layout()
fig.savefig(OUT_FIG / 'H4d_shrinkage.png')
plt.show()

### H4 Conclusion

H4 supports the main pair-level claim with an important caveat: distance explains part of the raw H1 gap, but it does not erase it. The connected-versus-unconnected contrast shrinks under distance matching, which is exactly what should happen if geometry is a real confound, but the contrast remains positive across the distance-controlled views.

This should shape the report language. We should not say that direct connectivity is independent of distance in a strong causal sense. The defensible statement is narrower and stronger: even after accounting for the fact that nearby neurons are more likely to be connected and more functionally similar, direct structural connectivity remains associated with higher signal correlation.


## 7.2 H5. Composition Controls

H5 asks whether H1 is actually driven by coarse composition. Pairs from the same area, layer, or cell type can be more similar for reasons that are not specific to direct synaptic connectivity. Conversely, across-area or across-type pairs may differ in both connection probability and functional response statistics.

The stratified contrasts retest H1 inside and across these coarse categories. The joint logistic model then asks whether `f_corr` remains informative after distance and composition indicators are entered together. This does not eliminate all possible biological confounds, but it checks the most obvious categorical structure available in the cohort metadata.


In [ ]:
# H5 (a) · stratified bootstrap + Mann-Whitney for each (feature, value) bucket
strat_rows = []
for col in ['same_area', 'same_layer', 'same_celltype']:
    for v in [True, False]:
        sub = pairs[pairs[col] == v]
        a = sub.loc[ sub.connected, 'f_corr'].to_numpy()
        b = sub.loc[~sub.connected, 'f_corr'].to_numpy()
        if len(a) < 5 or len(b) < 5:
            continue
        md_, ci_ = boot_mean_diff(a, b, n_boot=2000, seed=hash((col, v)) & 0xffff)
        _, p_   = mannwhitneyu(a, b, alternative='greater')
        strat_rows.append({'feature': col, 'value': v,
                           'n_conn': len(a), 'n_unconn': len(b),
                           'mean_diff': md_, 'ci_lo': ci_[0], 'ci_hi': ci_[1], 'p': p_})
strat = pd.DataFrame(strat_rows)
print(strat.to_string(index=False))
for r in strat.itertuples():
    RESULTS.append({
        'hypothesis': 'H5', 'test': f'stratum {r.feature}={r.value}',
        'mean_diff': r.mean_diff, 'ci_lo': r.ci_lo, 'ci_hi': r.ci_hi, 'p': r.p,
        'note': f'n_conn={r.n_conn}, n_unconn={r.n_unconn}',
    })

In [ ]:
# H5 (a) · forest plot — colour by feature, marker by within (●) / across (■)
feat_colors = {
    'same_area':     RAMP_FORE(0.30),
    'same_layer':    RAMP_FORE(0.55),
    'same_celltype': RAMP_FORE(0.78),
}
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ypos    = np.arange(len(strat))[::-1]
ylabels = []
for y_, r in zip(ypos, strat.itertuples()):
    feat_short = r.feature.replace('same_', '')
    where      = 'within' if r.value else 'across'
    marker     = 'o' if r.value else 's'
    color      = feat_colors[r.feature]
    ax.errorbar(r.mean_diff, y_,
                xerr=[[r.mean_diff - r.ci_lo], [r.ci_hi - r.mean_diff]],
                fmt=marker, color=color, markersize=8, capsize=4, lw=1.6,
                mec='white', mew=0.7)
    ax.text(r.ci_hi + 0.0015, y_, f'p={r.p:.1e}', va='center', fontsize=8.3, color='0.3')
    ylabels.append(f'{feat_short} · {where}\n(n_c={r.n_conn:,}, n_u={r.n_unconn:,})')
ax.axvline(0, color='0.4', lw=0.7, ls=':')
ax.set_yticks(ypos); ax.set_yticklabels(ylabels, fontsize=9)
ax.set_xlabel(r'$\Delta$ mean $F_{corr}$  (connected − unconnected)')
ax.set_title('H5(a) · Within- and across-stratum H1 retest')

# Generous left margin so labels fit, and a touch of right margin so p-values + legend
# coexist without overlap. The marker-shape legend lives in the bottom-right corner —
# the celltype-across point sits near x=0.03 with its p-value text at x≈0.045, so the
# region (x>0.058) is comfortably empty.
ax.set_xlim(-0.012, 0.072)
lg_h = [Line2D([0], [0], marker='o', color='w', markerfacecolor='0.3', markersize=8, label='within stratum'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='0.3', markersize=8, label='across stratum')]
ax.legend(handles=lg_h, loc='upper right', frameon=True,
          facecolor='white', edgecolor='0.85', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_FIG / 'H5a_stratified.png')
plt.show()

In [ ]:
# H5 (b) · joint logistic — z-score continuous regressors, keep categoricals as 0/1
Xj = pd.DataFrame({
    'f_corr':        zscore(pairs.f_corr.values),
    'dist_um':       zscore(pairs.dist_um.values),
    'same_area':     pairs.same_area.astype(int).values,
    'same_layer':    pairs.same_layer.astype(int).values,
    'same_celltype': pairs.same_celltype.astype(int).values,
})
Xj    = sm.add_constant(Xj)
yj    = pairs.connected.astype(int).to_numpy()
joint = sm.Logit(yj, Xj).fit(disp=False, cov_type='HC0')
print(joint.summary().tables[1])

RESULTS += [{
    'hypothesis': 'H5', 'test': 'joint logistic β(f_corr | dist, area, layer, celltype)',
    'mean_diff': float(joint.params['f_corr']),
    'ci_lo':     float(joint.params['f_corr'] - 1.96 * joint.bse['f_corr']),
    'ci_hi':     float(joint.params['f_corr'] + 1.96 * joint.bse['f_corr']),
    'p':         float(joint.pvalues['f_corr']),
    'note':      'standardised continuous; HC0 SE',
}]

In [ ]:
# H5 (b) · standardised coefficient bar chart, with f_corr highlighted
fig, ax = plt.subplots(figsize=(5.8, 3.9))
names = [r'$F_{corr}$', 'dist', 'same\narea', 'same\nlayer', 'same\ncell type']
keys  = ['f_corr', 'dist_um', 'same_area', 'same_layer', 'same_celltype']
betas = np.array([joint.params[k] for k in keys])
ses   = np.array([joint.bse[k]    for k in keys])
bar_c = ['#1F4E79' if k == 'f_corr' else '0.55' for k in keys]
ax.bar(np.arange(len(names)), betas, yerr=1.96 * ses,
       color=bar_c, edgecolor='white', capsize=4, width=0.65)
ax.axhline(0, color='0.4', lw=0.7, ls=':')
ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names, fontsize=9.5)
ax.set_ylabel(r'standardised logistic $\beta$  (±95% CI)')
ax.set_title('H5(b) · Joint logistic, HC0 SE')
yspan = (betas + 1.96 * ses).max() - (betas - 1.96 * ses).min()
pad   = 0.025 * yspan
for x, b_, s_ in zip(np.arange(len(names)), betas, ses):
    ytxt, va = (b_ + 1.96 * s_ + pad, 'bottom') if b_ >= 0 else (b_ - 1.96 * s_ - pad, 'top')
    ax.text(x, ytxt, f'{b_:+.2f}', ha='center', va=va, fontsize=8.6, color='0.25')
ax.margins(y=0.22)

# Empty quadrant: only the dist bar is deeply negative, so the lower-right region —
# below the small same_area / same_layer / same_celltype bars and above the dist trough
# — is comfortably empty. Anchor the readout there.
ax.text(0.97, 0.32,
        f'β($F_{{corr}}$) = {joint.params["f_corr"]:+.2f}\n'
        f'p = {joint.pvalues["f_corr"]:.1e}\n'
        f'(dominant continuous predictor)',
        transform=ax.transAxes, va='center', ha='right', fontsize=8.6, color='#1F4E79',
        bbox=dict(boxstyle='round,pad=0.32', facecolor='white', edgecolor='0.75', lw=0.7))
fig.tight_layout()
fig.savefig(OUT_FIG / 'H5b_joint_logistic.png')
plt.show()

### H5 Conclusion

H5 supports the robustness of H1. The connected-pair effect remains positive within the coarse composition splits and in the joint model that includes distance plus area, layer, and cell-type indicators. That makes it unlikely that the H1 result is only a trivial consequence of comparing same-area or same-class pairs against mixed pairs.

The result should still be described carefully. Area, layer, and cell-type labels are coarse, and pair rows remain statistically dependent because neurons appear in many pairs. H5 strengthens the H1 interpretation, but it does not prove that every possible anatomical or transcriptional covariate has been controlled.


## 7.3 H6. Orientation Similarity

H6 asks how visual tuning relates to the structure-function signal. In visual cortex, neurons with similar orientation preference may be more likely to connect, and they may also have higher signal correlation. Orientation similarity is therefore both scientifically meaningful and a possible explanation for part of H1.

This section focuses on the well-tuned subset, because orientation similarity is interpretable only when both neurons have reliable orientation tuning. The marginal test asks whether connected pairs are more orientation-similar. The joint model then asks whether `F_corr` still contributes once orientation similarity and distance are included together.


In [ ]:
# H6 (a) · marginal Δ ori_sim on the well-tuned subset
GOSI_MIN = 0.10
sub = pairs.dropna(subset=['ori_sim']).copy()
sub = sub[sub.gOSI_min > GOSI_MIN]
print(f'well-tuned pairs (gOSI_min > {GOSI_MIN}): {len(sub):,}  ({len(sub)/len(pairs):.1%} of all pairs)')

a_o = sub.loc[ sub.connected, 'ori_sim'].to_numpy()
b_o = sub.loc[~sub.connected, 'ori_sim'].to_numpy()
_, p_o     = mannwhitneyu(a_o, b_o, alternative='greater')
md_o, ci_o = boot_mean_diff(a_o, b_o, n_boot=4000, seed=1)
print(f'ori_sim  Δmean = {md_o:+.4f}   CI [{ci_o[0]:+.4f}, {ci_o[1]:+.4f}]   p = {p_o:.2e}')
print(f'  connected  : n={len(a_o):,}, mean ori_sim = {a_o.mean():+.4f}')
print(f'  unconnected: n={len(b_o):,}, mean ori_sim = {b_o.mean():+.4f}')

RESULTS += [{
    'hypothesis': 'H6', 'test': 'marginal Δ ori_sim (connected − unconnected)',
    'mean_diff': md_o, 'ci_lo': ci_o[0], 'ci_hi': ci_o[1], 'p': p_o,
    'note': f'gOSI_min > {GOSI_MIN};  n_conn={len(a_o)}, n_unconn={len(b_o)}',
}]

In [ ]:
# H6 (a) · ori_sim distribution by connection status (filled unconnected, stepped connected)
fig, ax = plt.subplots(figsize=(5.6, 3.7))
bins = np.linspace(-1.0, 1.0, 41)
ax.hist(b_o, bins=bins, density=True, histtype='stepfilled',
        color=PAL_CONN['none'], alpha=0.55, edgecolor=PAL_CONN['none'],
        label=f'unconnected (n={len(b_o):,})')
ax.hist(a_o, bins=bins, density=True, histtype='step', linewidth=2.2,
        color=PAL_CONN['uni'], label=f'connected (n={len(a_o):,})')
ax.axvline(b_o.mean(), color=PAL_CONN['none'], lw=1.0, ls='--', alpha=0.85)
ax.axvline(a_o.mean(), color=PAL_CONN['uni'],  lw=1.0, ls='--', alpha=0.85)
ax.axvline(0,           color='0.45', lw=0.6, ls=':')
ax.set_xlabel(r'orientation similarity  $\cos(2\Delta\theta)$')
ax.set_ylabel('density')
ax.set_title(f'H6(a) · Connected pairs are slightly more co-tuned (gOSI_min > {GOSI_MIN})')
# Bottom centre is empty in this U-shaped distribution — perfect spot for the stats box.
ax.text(0.50, 0.05,
        f'Δmean = {md_o:+.3f}    p = {p_o:.1e}\nCI [{ci_o[0]:+.3f}, {ci_o[1]:+.3f}]',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=8.8, color='0.25',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='0.75', lw=0.7))
ax.legend(loc='upper center', ncol=1, bbox_to_anchor=(0.50, 0.98))
fig.tight_layout()
fig.savefig(OUT_FIG / 'H6a_ori_sim_dist.png')
plt.show()

In [ ]:
# H6 (b) · joint logistic on the well-tuned subset
Xs = pd.DataFrame({
    'f_corr':  zscore(sub.f_corr.values),
    'ori_sim': zscore(sub.ori_sim.values),
    'dist_um': zscore(sub.dist_um.values),
})
Xs = sm.add_constant(Xs)
ys = sub.connected.astype(int).to_numpy()
m6 = sm.Logit(ys, Xs).fit(disp=False, cov_type='HC0')
print(m6.summary().tables[1])

RESULTS += [
    {'hypothesis': 'H6', 'test': 'joint logistic β(f_corr | ori_sim, dist)',
     'mean_diff': float(m6.params['f_corr']),
     'ci_lo':     float(m6.params['f_corr'] - 1.96 * m6.bse['f_corr']),
     'ci_hi':     float(m6.params['f_corr'] + 1.96 * m6.bse['f_corr']),
     'p':         float(m6.pvalues['f_corr']),
     'note':      'standardised; well-tuned subset'},
    {'hypothesis': 'H6', 'test': 'joint logistic β(ori_sim | f_corr, dist)',
     'mean_diff': float(m6.params['ori_sim']),
     'ci_lo':     float(m6.params['ori_sim'] - 1.96 * m6.bse['ori_sim']),
     'ci_hi':     float(m6.params['ori_sim'] + 1.96 * m6.bse['ori_sim']),
     'p':         float(m6.pvalues['ori_sim']),
     'note':      'standardised; well-tuned subset'},
]

In [ ]:
# H6 (b) · standardised coefficient bar chart
fig, ax = plt.subplots(figsize=(5.4, 3.9))
names = [r'$F_{corr}$', 'ori_sim', 'dist']
keys  = ['f_corr', 'ori_sim', 'dist_um']
betas = np.array([m6.params[k] for k in keys])
ses   = np.array([m6.bse[k]    for k in keys])
bar_c = [PAL_CONN['uni'], '#1F77B4', '0.55']
ax.bar(np.arange(3), betas, yerr=1.96 * ses,
       color=bar_c, edgecolor='white', capsize=5, width=0.6)
ax.axhline(0, color='0.4', lw=0.7, ls=':')
ax.set_xticks(np.arange(3)); ax.set_xticklabels(names)
ax.set_ylabel(r'standardised logistic $\beta$  (±95% CI)')
ax.set_title(f'H6(b) · Joint logistic (well-tuned, gOSI_min > {GOSI_MIN})')
yspan = (betas + 1.96 * ses).max() - (betas - 1.96 * ses).min()
pad   = 0.030 * yspan
for x, b_, s_ in zip(np.arange(3), betas, ses):
    ytxt, va = (b_ + 1.96 * s_ + pad, 'bottom') if b_ >= 0 else (b_ - 1.96 * s_ - pad, 'top')
    ax.text(x, ytxt, f'β = {b_:+.2f}', ha='center', va=va, fontsize=8.8, color='0.25')
ax.margins(y=0.24)

# Big empty quadrant: below the F_corr / ori_sim bars (both small) and to the LEFT of
# the dist trough. Lower-left of the axes is clean.
ratio = betas[0] / betas[1] if betas[1] != 0 else float('inf')
ax.text(0.04, 0.30,
        f'$F_{{corr}}$ carries\n≈{ratio:.1f}× the weight\nof ori_sim',
        transform=ax.transAxes, va='center', ha='left', fontsize=8.8, color=PAL_CONN['uni'],
        bbox=dict(boxstyle='round,pad=0.34', facecolor='white', edgecolor='0.75', lw=0.7))
fig.tight_layout()
fig.savefig(OUT_FIG / 'H6b_joint_logistic.png')
plt.show()

In [ ]:
summary = pd.DataFrame(RESULTS)[['hypothesis', 'test', 'mean_diff', 'ci_lo', 'ci_hi', 'p', 'note']]
summary.to_csv(OUT_TBL / 'person2_h4_h6_controls_summary.csv', index=False)
print(f'saved -> {OUT_TBL / "person2_h4_h6_controls_summary.csv"}')
summary

### H6 Conclusion

H6 supports a layered interpretation. Connected pairs are slightly more orientation-similar among well-tuned neurons, which is consistent with like-to-like wiring in visual cortex. However, signal correlation remains the stronger predictor when orientation similarity and distance are modeled together.

The best interpretation is that orientation preference captures one specific tuning axis inside a broader response-similarity relationship. Orientation similarity contributes useful biological context, but it does not replace the functional correlation result. For the report, this section should be used to say that H1 is not merely an orientation-tuning artifact, while still acknowledging that tuning similarity is part of the structure-function story.


## 8. Network Analysis and Topology Comparison, Including H7

The assignment asks not only whether directly connected neurons are correlated, but also whether the structural and functional networks have similar topology. This is a harder comparison because the two networks have different mathematical forms: the structural graph is sparse, directed, and synapse-weighted, while the functional matrix is dense, symmetric, and signed.

H7 handles the most interpretable topology comparison: node centrality. If the networks have similar topology, neurons that are structural hubs should also be functionally coupled to many other neurons or have high mean functional association with the cohort. This section therefore computes structural degree and strength, functional hub metrics, partial rank correlations, and thresholded graph sensitivities.


In [ ]:
def mean_pairwise_distance_um(xyz):
    dist = np.linalg.norm(xyz[:, None, :] - xyz[None, :, :], axis=2)
    np.fill_diagonal(dist, np.nan)
    return np.nanmean(dist, axis=1)

xyz = matched[['pt_position_x', 'pt_position_y', 'pt_position_z']].to_numpy(dtype=float)
mean_dist_um = mean_pairwise_distance_um(xyz)

F_corr_no_diag = F_corr.copy()
np.fill_diagonal(F_corr_no_diag, np.nan)

mean_fcorr_signed = np.nanmean(F_corr_no_diag, axis=1)
mean_fcorr_pos_clipped = np.nanmean(np.clip(F_corr_no_diag, 0, None), axis=1)
mean_fcorr_abs = np.nanmean(np.abs(F_corr_no_diag), axis=1)

iu_all, ju_all = np.triu_indices(N, k=1)
pair_dist = np.linalg.norm(xyz[iu_all] - xyz[ju_all], axis=1)
pair_fc = F_corr_no_diag[iu_all, ju_all]
pair_mask = np.isfinite(pair_fc) & valid[iu_all] & valid[ju_all]
d_z = (pair_dist[pair_mask] - pair_dist[pair_mask].mean()) / pair_dist[pair_mask].std(ddof=0)
X_pair = np.column_stack([np.ones(d_z.size), d_z, d_z ** 2])
beta_pair = np.linalg.lstsq(X_pair, pair_fc[pair_mask], rcond=None)[0]
pair_resid = pair_fc[pair_mask] - X_pair @ beta_pair
F_corr_pairdist_resid = np.full((N, N), np.nan, dtype=float)
ii, jj = iu_all[pair_mask], ju_all[pair_mask]
F_corr_pairdist_resid[ii, jj] = pair_resid
F_corr_pairdist_resid[jj, ii] = pair_resid
mean_fcorr_pairdist_resid = np.nanmean(F_corr_pairdist_resid, axis=1)
print(f'pairwise distance residual model: F_corr ~ dist + dist^2, n_pairs={pair_mask.sum():,}')

area_arr = matched.brain_area.to_numpy()
mean_fcorr_same_area = np.full(N, np.nan, dtype=float)
mean_dist_same_area = np.full(N, np.nan, dtype=float)
dist_matrix = np.linalg.norm(xyz[:, None, :] - xyz[None, :, :], axis=2)
np.fill_diagonal(dist_matrix, np.nan)
for i in range(N):
    same = area_arr == area_arr[i]
    same[i] = False
    mean_fcorr_same_area[i] = np.nanmean(F_corr_no_diag[i, same])
    mean_dist_same_area[i] = np.nanmean(dist_matrix[i, same])

rho_resid_dist, p_resid_dist = spearmanr(mean_fcorr_pairdist_resid, mean_dist_um, nan_policy='omit')
print(f'sanity: mean pair-distance residual still correlates with mean distance: rho={rho_resid_dist:+.3f}, p={p_resid_dist:.2e}')
if np.isfinite(rho_resid_dist) and abs(rho_resid_dist) > 0.10:
    print('diagnostic warning: node-averaged pair-distance residuals still carry distance structure; treat this as an incomplete distance-control check.')

hub_metrics = pd.DataFrame({
    'matrix_index': np.arange(N),
    'pt_root_id': matched.pt_root_id.values,
    'brain_area': matched.brain_area.values,
    'layer': matched.layer.values,
    'cell_type': matched.cell_type.values,
    'valid_function': valid,
    'x_um': matched.pt_position_x.values,
    'y_um': matched.pt_position_y.values,
    'z_um': matched.pt_position_z.values,
    'mean_dist_um': mean_dist_um,
    'out_degree': np.asarray(A_struct.sum(axis=1)).ravel().astype(float),
    'in_degree': np.asarray(A_struct.sum(axis=0)).ravel().astype(float),
    'out_strength': np.asarray(C_strength.sum(axis=1)).ravel(),
    'in_strength': np.asarray(C_strength.sum(axis=0)).ravel(),
    'mean_fcorr_signed': mean_fcorr_signed,
    'mean_fcorr_pos_clipped': mean_fcorr_pos_clipped,
    'mean_fcorr_abs': mean_fcorr_abs,
    'mean_fcorr_pairdist_resid': mean_fcorr_pairdist_resid,
    'mean_fcorr_same_area': mean_fcorr_same_area,
    'mean_dist_same_area': mean_dist_same_area,
})

hub_metrics['total_degree'] = hub_metrics.out_degree + hub_metrics.in_degree
hub_metrics['total_strength'] = hub_metrics.out_strength + hub_metrics.in_strength

out_metrics = OUT_TBL / 'h7_hub_metrics.csv'
hub_metrics.to_csv(out_metrics, index=False)
print(f'saved: {out_metrics}')
hub_metrics.describe(include='all').T.loc[
    ['total_degree', 'total_strength', 'mean_fcorr_signed', 'mean_fcorr_pairdist_resid', 'mean_fcorr_same_area', 'mean_dist_um'],
    ['count', 'mean', 'std', 'min', '50%', 'max']
]


In [ ]:
def _control_matrix(control_df):
    if control_df is None or control_df.shape[1] == 0:
        return np.empty((0, 0))

    parts = []
    for col in control_df.columns:
        s = control_df[col]
        if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            values = s.astype(float).to_numpy()
            if pd.Series(values).nunique(dropna=True) > 2:
                values = stats.rankdata(values)
            parts.append(values)
        else:
            dummies = pd.get_dummies(s, prefix=col, drop_first=True, dtype=float)
            for dcol in dummies.columns:
                parts.append(dummies[dcol].to_numpy(dtype=float))

    if not parts:
        return np.empty((len(control_df), 0))
    Z = np.column_stack(parts)
    keep = np.nanstd(Z, axis=0) > 0
    return Z[:, keep]


def partial_spearman(x, y, control_df=None):
    base = pd.DataFrame({'x': x, 'y': y}).reset_index(drop=True)
    if control_df is not None and control_df.shape[1] > 0:
        controls = control_df.reset_index(drop=True)
        base = pd.concat([base, controls], axis=1)
    mask = base.notna().all(axis=1).to_numpy()
    base = base.loc[mask].reset_index(drop=True)

    rx = stats.rankdata(base['x'].to_numpy(dtype=float))
    ry = stats.rankdata(base['y'].to_numpy(dtype=float))

    if control_df is None or control_df.shape[1] == 0:
        r, p = pearsonr(rx, ry)
        return r, p, len(base)

    Z = _control_matrix(base.drop(columns=['x', 'y']))
    if Z.shape[1] == 0:
        r, p = pearsonr(rx, ry)
        return r, p, len(base)

    Z = (Z - Z.mean(axis=0)) / Z.std(axis=0, ddof=0)
    X = np.column_stack([np.ones(len(base)), Z])
    bx = np.linalg.lstsq(X, rx, rcond=None)[0]
    by = np.linalg.lstsq(X, ry, rcond=None)[0]
    ex = rx - X @ bx
    ey = ry - X @ by
    r, p = pearsonr(ex, ey)
    return r, p, len(base)


def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    mask = np.isfinite(p)
    if not mask.any():
        return q
    pv = p[mask]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / np.arange(1, len(pv) + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(pv)
    out[order] = np.minimum(adjusted, 1.0)
    q[mask] = out
    return q


def h7_corr(df, smetric, fmetric, controls):
    sub = df[df.valid_function].copy()
    if controls == 'raw':
        mask = sub[[smetric, fmetric]].notna().all(axis=1)
        rho, p = spearmanr(sub.loc[mask, smetric], sub.loc[mask, fmetric])
        return float(rho), float(p), int(mask.sum())
    control_sets = {
        'distance': ['mean_dist_um'],
        'distance+area': ['mean_dist_um', 'brain_area'],
        'distance+area+cell_type': ['mean_dist_um', 'brain_area', 'cell_type'],
        'distance+area+layer': ['mean_dist_um', 'brain_area', 'layer'],
        'total_degree': ['total_degree'],
        'distance+area+total_degree': ['mean_dist_um', 'brain_area', 'total_degree'],
    }
    rho, p, n = partial_spearman(
        sub[smetric].to_numpy(dtype=float),
        sub[fmetric].to_numpy(dtype=float),
        sub[control_sets[controls]],
    )
    return float(rho), float(p), int(n)


def bootstrap_h7_ci(df, smetric, fmetric, controls, n_boot=1000, seed=7):
    rng = np.random.default_rng(seed)
    sub = df[df.valid_function].reset_index(drop=True).copy()
    boot = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        sample = sub.iloc[rng.integers(0, len(sub), len(sub))].reset_index(drop=True)
        boot[b] = h7_corr(sample, smetric, fmetric, controls)[0]
    return float(np.nanpercentile(boot, 2.5)), float(np.nanpercentile(boot, 97.5))


def stratified_permutation_p(df, smetric, fmetric, controls, strata_cols=('brain_area', 'layer'), n_perm=10000, seed=11):
    rng = np.random.default_rng(seed)
    sub = df[df.valid_function].reset_index(drop=True).copy()
    observed = h7_corr(sub, smetric, fmetric, controls)[0]
    perm = np.empty(n_perm, dtype=float)
    group_indices = [idx.to_numpy() for _, idx in sub.groupby(list(strata_cols), observed=False).groups.items()]
    y0 = sub[fmetric].to_numpy().copy()
    for b in range(n_perm):
        y = y0.copy()
        for idx in group_indices:
            if len(idx) > 1:
                y[idx] = rng.permutation(y[idx])
        sample = sub.copy()
        sample[fmetric] = y
        perm[b] = h7_corr(sample, smetric, fmetric, controls)[0]
    p = (np.sum(np.abs(perm) >= abs(observed)) + 1) / (n_perm + 1)
    return float(observed), float(p)


def run_h7_tests(df, cohort_label):
    structural_metrics = [
        'total_degree', 'total_strength',
        'in_degree', 'out_degree',
        'in_strength', 'out_strength',
    ]
    functional_metrics = [
        'mean_fcorr_signed',
        'mean_fcorr_pos_clipped',
        'mean_fcorr_abs',
    ]
    control_sets = {
        'raw': [],
        'distance': ['mean_dist_um'],
        'distance+area': ['mean_dist_um', 'brain_area'],
        'distance+area+cell_type': ['mean_dist_um', 'brain_area', 'cell_type'],
        'distance+area+layer': ['mean_dist_um', 'brain_area', 'layer'],
    }

    rows = []
    for fmetric in functional_metrics:
        for smetric in structural_metrics:
            for control_label, controls in control_sets.items():
                sub = df[df.valid_function].copy()
                if control_label == 'raw':
                    mask = sub[[smetric, fmetric]].notna().all(axis=1)
                    rho, p = spearmanr(sub.loc[mask, smetric], sub.loc[mask, fmetric])
                    n = int(mask.sum())
                else:
                    rho, p, n = partial_spearman(
                        sub[smetric].to_numpy(dtype=float),
                        sub[fmetric].to_numpy(dtype=float),
                        sub[controls],
                    )
                rows.append({
                    'cohort': cohort_label,
                    'functional_metric': fmetric,
                    'structural_metric': smetric,
                    'controls': control_label,
                    'test_role': (
                        'primary'
                        if (
                            cohort_label == 'all_906'
                            and fmetric == 'mean_fcorr_signed'
                            and smetric == 'total_strength'
                            and control_label == 'distance+area'
                        )
                        else 'strict_layer_sensitivity'
                        if (
                            cohort_label == 'all_906'
                            and fmetric == 'mean_fcorr_signed'
                            and smetric in ['total_degree', 'total_strength']
                            and control_label == 'distance+area+layer'
                        )
                        else 'exploratory'
                    ),
                    'rho': float(rho),
                    'p_value': float(p),
                    'n_neurons': n,
                })
    return pd.DataFrame(rows)

summary_all = run_h7_tests(hub_metrics, 'all_906')
h7_summary = summary_all.copy()
h7_summary['rank_variance_approx'] = h7_summary['rho'] ** 2
h7_summary['q_bh_all_tests'] = bh_fdr(h7_summary.p_value)

bootstrap_specs = [
    ('total_strength', 'mean_fcorr_signed', 'distance+area', 'primary'),
    ('total_strength', 'mean_fcorr_signed', 'distance+area+layer', 'strict layer'),
    ('total_strength', 'mean_fcorr_signed', 'distance+area+total_degree', 'strength beyond degree'),
    ('total_degree', 'mean_fcorr_signed', 'distance+area+layer', 'strict layer'),
    ('in_strength', 'mean_fcorr_signed', 'distance+area', 'directional'),
    ('out_strength', 'mean_fcorr_signed', 'distance+area', 'directional'),
    ('total_strength', 'mean_fcorr_pairdist_resid', 'raw', 'pairwise-distance residual'),
    ('total_strength', 'mean_fcorr_abs', 'distance+area', 'absolute-coupling check'),
]
boot_rows = []
for smetric, fmetric, controls, role in bootstrap_specs:
    rho, p, n = h7_corr(hub_metrics, smetric, fmetric, controls)
    ci_lo, ci_hi = bootstrap_h7_ci(hub_metrics, smetric, fmetric, controls, n_boot=1000)
    boot_rows.append({
        'role': role,
        'cohort': 'all_906',
        'functional_metric': fmetric,
        'structural_metric': smetric,
        'controls': controls,
        'rho': rho,
        'p_value': p,
        'rank_variance_approx': rho ** 2,
        'ci_low_bootstrap': ci_lo,
        'ci_high_bootstrap': ci_hi,
        'n_neurons': n,
    })
h7_bootstrap = pd.DataFrame(boot_rows)
out_bootstrap = OUT_TBL / 'h7_key_bootstrap_ci.csv'
h7_bootstrap.to_csv(out_bootstrap, index=False)
print(f'saved: {out_bootstrap}')

perm_specs = [
    ('total_strength', 'mean_fcorr_signed', 'distance+area'),
    ('total_strength', 'mean_fcorr_signed', 'distance+area+layer'),
    ('in_strength', 'mean_fcorr_signed', 'distance+area'),
    ('out_strength', 'mean_fcorr_signed', 'distance+area'),
]
N_STRATIFIED_PERMUTATIONS = 10_000
perm_rows = []
for smetric, fmetric, controls in perm_specs:
    rho_obs, p_perm = stratified_permutation_p(
        hub_metrics, smetric, fmetric, controls,
        strata_cols=('brain_area', 'layer'),
        n_perm=N_STRATIFIED_PERMUTATIONS,
    )
    perm_rows.append({
        'cohort': 'all_906',
        'functional_metric': fmetric,
        'structural_metric': smetric,
        'controls': controls,
        'strata_preserved': 'brain_area+layer',
        'rho': rho_obs,
        'rank_variance_approx': rho_obs ** 2,
        'p_stratified_permutation': p_perm,
        'n_permutations': N_STRATIFIED_PERMUTATIONS,
    })
h7_permutation = pd.DataFrame(perm_rows)
out_permutation = OUT_TBL / 'h7_stratified_permutation.csv'
h7_permutation.to_csv(out_permutation, index=False)
print(f'saved: {out_permutation}')

def custom_h7_row(df, label, smetric, fmetric, control_cols):
    sub = df[df.valid_function].copy() if 'valid_function' in df.columns else df.copy()
    cols = [smetric, fmetric] + list(control_cols)
    sub = sub.dropna(subset=cols).reset_index(drop=True)
    if control_cols:
        rho, p, n = partial_spearman(
            sub[smetric].to_numpy(dtype=float),
            sub[fmetric].to_numpy(dtype=float),
            sub[list(control_cols)],
        )
    else:
        rho, p = spearmanr(sub[smetric], sub[fmetric])
        n = len(sub)
    return {
        'analysis': label,
        'structural_metric': smetric,
        'functional_metric': fmetric,
        'controls': '+'.join(control_cols) if control_cols else 'raw',
        'rho': float(rho),
        'rank_variance_approx': float(rho ** 2),
        'p_value': float(p),
        'n_neurons': int(n),
    }

def recompute_subset_metrics(mask, subset_label):
    idx = np.flatnonzero(mask)
    sub = hub_metrics.iloc[idx].copy().reset_index(drop=True)
    F_sub = F_corr_no_diag[np.ix_(idx, idx)].copy()
    D_sub = dist_matrix[np.ix_(idx, idx)].copy()
    C_sub = C_strength[idx][:, idx]
    A_sub = C_sub != 0
    sub['subset_label'] = subset_label
    sub['mean_fcorr_subset'] = np.nanmean(F_sub, axis=1)
    sub['mean_dist_subset'] = np.nanmean(D_sub, axis=1)
    sub['subset_total_strength'] = np.asarray(C_sub.sum(axis=1)).ravel() + np.asarray(C_sub.sum(axis=0)).ravel()
    sub['subset_total_degree'] = np.asarray(A_sub.sum(axis=1)).ravel() + np.asarray(A_sub.sum(axis=0)).ravel()
    return sub

robust_rows = []
robust_rows.append(custom_h7_row(hub_metrics, 'strength beyond degree', 'total_strength', 'mean_fcorr_signed', ['total_degree']))
robust_rows.append(custom_h7_row(hub_metrics, 'strength beyond degree', 'total_strength', 'mean_fcorr_signed', ['mean_dist_um', 'brain_area', 'total_degree']))
robust_rows.append(custom_h7_row(hub_metrics, 'cell-type strict control', 'total_strength', 'mean_fcorr_signed', ['mean_dist_um', 'brain_area', 'cell_type']))
robust_rows.append(custom_h7_row(hub_metrics, 'same-area functional mean', 'total_strength', 'mean_fcorr_same_area', []))
robust_rows.append(custom_h7_row(hub_metrics, 'same-area functional mean', 'total_strength', 'mean_fcorr_same_area', ['mean_dist_same_area', 'brain_area']))
robust_rows.append(custom_h7_row(hub_metrics, 'same-area functional mean', 'total_strength', 'mean_fcorr_same_area', ['mean_dist_same_area', 'brain_area', 'layer']))
robust_rows.append(custom_h7_row(hub_metrics, 'pair-distance residual diagnostic', 'mean_fcorr_pairdist_resid', 'mean_dist_um', []))
robust_rows.append(custom_h7_row(hub_metrics, 'pair-distance residual diagnostic', 'total_strength', 'mean_fcorr_pairdist_resid', []))

v1_recomputed = recompute_subset_metrics((hub_metrics.brain_area == 'V1').to_numpy(), 'V1')
robust_rows.append(custom_h7_row(v1_recomputed, 'V1 recomputed functional mean; full structural strength', 'total_strength', 'mean_fcorr_subset', []))
robust_rows.append(custom_h7_row(v1_recomputed, 'V1 recomputed functional mean; full structural strength', 'total_strength', 'mean_fcorr_subset', ['mean_dist_subset']))
robust_rows.append(custom_h7_row(v1_recomputed, 'V1 recomputed functional mean; full structural strength', 'total_strength', 'mean_fcorr_subset', ['mean_dist_subset', 'layer']))
robust_rows.append(custom_h7_row(v1_recomputed, 'V1 induced-subgraph strength', 'subset_total_strength', 'mean_fcorr_subset', []))
robust_rows.append(custom_h7_row(v1_recomputed, 'V1 induced-subgraph strength', 'subset_total_strength', 'mean_fcorr_subset', ['mean_dist_subset', 'layer']))

h7_robustness = pd.DataFrame(robust_rows)
out_robustness = OUT_TBL / 'h7_robustness_diagnostics.csv'
h7_robustness.to_csv(out_robustness, index=False)
print(f'saved: {out_robustness}')

strength_quartiles = hub_metrics.copy()
strength_quartiles['strength_quartile'] = pd.qcut(strength_quartiles.total_strength, 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
strength_quartile_summary = strength_quartiles.groupby('strength_quartile', observed=False).agg(
    n_neurons=('matrix_index', 'size'),
    mean_total_strength=('total_strength', 'mean'),
    mean_total_degree=('total_degree', 'mean'),
    mean_signed_fcorr=('mean_fcorr_signed', 'mean'),
    mean_abs_fcorr=('mean_fcorr_abs', 'mean'),
).reset_index()
out_quartiles = OUT_TBL / 'h7_strength_quartiles.csv'
strength_quartile_summary.to_csv(out_quartiles, index=False)
print(f'saved: {out_quartiles}')

out_summary = OUT_TBL / 'h7_summary.csv'
h7_summary.to_csv(out_summary, index=False)
print(f'saved: {out_summary}')

primary_row = h7_summary[
    (h7_summary.cohort == 'all_906')
    & (h7_summary.functional_metric == 'mean_fcorr_signed')
    & (h7_summary.structural_metric == 'total_strength')
    & (h7_summary.controls == 'distance+area')
].iloc[0]
print(
    'primary narrow signed-alignment effect: '
    f"rho={primary_row.rho:+.3f}, rank_variance_approx={primary_row.rank_variance_approx:.4f} "
    f"({100 * primary_row.rank_variance_approx:.1f}% rank variance)"
)

primary = h7_summary[
    (h7_summary.cohort == 'all_906')
    & (h7_summary.functional_metric == 'mean_fcorr_signed')
    & (h7_summary.structural_metric.isin(['total_degree', 'total_strength', 'in_strength', 'out_strength']))
    & (h7_summary.controls.isin(['raw', 'distance', 'distance+area', 'distance+area+cell_type', 'distance+area+layer']))
].copy()
primary.sort_values(['structural_metric', 'controls'])


In [ ]:
plot_df = hub_metrics[hub_metrics.valid_function].copy()
coef_df = h7_summary[
    (h7_summary.cohort == 'all_906')
    & (h7_summary.functional_metric == 'mean_fcorr_signed')
    & (h7_summary.structural_metric.isin(['total_degree', 'total_strength']))
    & (h7_summary.controls.isin(['raw', 'distance+area', 'distance+area+cell_type', 'distance+area+layer']))
].copy()
coef_df['metric_label'] = coef_df.structural_metric.map({
    'total_degree': 'degree',
    'total_strength': 'strength',
})
coef_df['control_label'] = coef_df.controls.map({
    'raw': 'raw',
    'distance+area': 'partial: dist+area',
    'distance+area+cell_type': 'strict: dist+area+cell type',
    'distance+area+layer': 'strict: dist+area+layer',
})

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5))
ax = axes[0, 0]
sns.histplot(plot_df.total_degree, bins=35, color='#4E79A7', ax=ax)
ax.set(title='Structural hub distribution', xlabel='total degree (in + out)', ylabel='neurons')

for ax, xcol, xlabel in [
    (axes[0, 1], 'total_degree', 'total degree (in + out)'),
    (axes[1, 0], 'total_strength', 'log10 summed synapse size + 1'),
]:
    x = plot_df[xcol].to_numpy(dtype=float)
    x_plot = np.log10(x + 1) if xcol == 'total_strength' else x
    for area_name, sub in plot_df.groupby('brain_area'):
        sx = sub[xcol].to_numpy(dtype=float)
        sx = np.log10(sx + 1) if xcol == 'total_strength' else sx
        ax.scatter(
            sx,
            sub.mean_fcorr_signed,
            s=22,
            alpha=0.78,
            linewidths=0.25,
            edgecolor='white',
            color=PAL_AREA.get(area_name, '0.45'),
            label=area_name,
        )
    trend = pd.DataFrame({'x': x_plot, 'y': plot_df.mean_fcorr_signed}).copy()
    trend['bin'] = pd.qcut(trend.x, 6, duplicates='drop')
    trend = trend.groupby('bin', observed=False).agg(x=('x', 'median'), y=('y', 'median')).reset_index(drop=True)
    ax.plot(trend.x, trend.y, color='0.15', lw=1.8, marker='o', ms=4, label='bin median')
    rho = coef_df[(coef_df.structural_metric == xcol) & (coef_df.controls == 'raw')].rho.iloc[0]
    p = coef_df[(coef_df.structural_metric == xcol) & (coef_df.controls == 'raw')].p_value.iloc[0]
    ax.set(title=f'Signed population alignment: rho={rho:+.3f}, p={p:.1e}', xlabel=xlabel, ylabel='mean signed F_corr')
    ax.legend(title='area', frameon=False, loc='best')

ax = axes[1, 1]
sns.barplot(
    data=coef_df,
    x='metric_label',
    y='rho',
    hue='control_label',
    palette=['#7f8c8d', '#4E79A7', '#F28E2B', '#E15759'],
    ax=ax,
)
ax.axhline(0, color='0.2', lw=0.8)
ax.set(title='H7 rank correlations', xlabel='', ylabel='Spearman / partial Spearman rho')
ax.legend(title='', frameon=False)

fig.suptitle('H7: signed F_corr association is weak and layer-sensitive', y=1.01, fontsize=14, fontweight='semibold')
plt.tight_layout()
out_fig = OUT_FIG / 'h7_hub_coupling.png'
fig.savefig(out_fig, bbox_inches='tight')
print(f'saved: {out_fig}')
plt.show()


### H7 Hub-Coupling Conclusion

H7 gives limited support for topology-level alignment. The narrow signed relationship between structural strength and mean signed signal correlation is detectable but weak, and it changes under stricter controls and alternative functional definitions. This is very different from H1, where the pair-level connected-versus-unconnected contrast is robust and directly answers the assignment's first question.

The report should therefore avoid claiming that structural hubs are clearly functional hubs. A more defensible conclusion is that there is a small signed population-alignment signal among more structurally connected or stronger neurons, but this does not generalize cleanly across all hub metrics or functional graph definitions. H7 belongs in the network-analysis section as a cautious topology comparison, not as the main result.


## 8.1 H7 Sensitivity View

These tables expose how the H7 result changes under alternative functional metrics and stricter controls. This matters because topology conclusions are much more sensitive to analysis choices than the H1 pair-level comparison.

The key distinction is between signed mean correlation, positive-clipped correlation, absolute correlation, and thresholded functional degree. A result that appears only for one functional definition should be treated as a limited topology signal rather than broad structural-functional network equivalence.


In [ ]:
sensitivity_view = h7_summary[
    (h7_summary.structural_metric.isin(['total_degree', 'total_strength', 'in_degree', 'out_degree', 'in_strength', 'out_strength']))
    & (h7_summary.controls.isin(['raw', 'distance', 'distance+area', 'distance+area+cell_type', 'distance+area+layer']))
].copy()
sensitivity_view = sensitivity_view.sort_values([
    'functional_metric', 'cohort', 'structural_metric', 'controls'
])
key_sensitivity = sensitivity_view[
    sensitivity_view.functional_metric.isin(['mean_fcorr_signed', 'mean_fcorr_abs'])
    & sensitivity_view.structural_metric.isin(['total_degree', 'total_strength', 'in_strength', 'out_strength'])
    & sensitivity_view.controls.isin(['distance+area', 'distance+area+cell_type', 'distance+area+layer'])
].copy()
print(key_sensitivity.to_string(index=False))
h7_robustness


## 8.2 Thresholded Topology Sensitivity

A dense correlation matrix is not automatically comparable to a sparse synaptic graph. To make a graph-level comparison, the functional matrix has to be thresholded, and the resulting topology can depend strongly on that threshold. This section checks that sensitivity directly.

The functional graph is thresholded at densities around the structural density. The analysis then compares degree and strength coupling, and contrasts average clustering in the structural and thresholded functional graphs. These outputs should be read as exploratory topology diagnostics rather than as definitive evidence of network equivalence.


In [ ]:
def top_k_functional_adjacency(F_corr, k, valid_mask=None, positive_only=True):
    n = F_corr.shape[0]
    iu, ju = np.triu_indices(n, k=1)
    vals = F_corr[iu, ju]
    mask = np.isfinite(vals)
    if valid_mask is not None:
        mask &= valid_mask[iu] & valid_mask[ju]
    if positive_only:
        mask &= vals > 0
    iu, ju, vals = iu[mask], ju[mask], vals[mask]
    k = int(min(k, len(vals)))
    if k <= 0:
        return np.zeros((n, n), dtype=bool), np.full((n, n), np.nan)
    keep = np.argpartition(vals, -k)[-k:]
    A = np.zeros((n, n), dtype=bool)
    W = np.full((n, n), np.nan)
    A[iu[keep], ju[keep]] = True
    A[ju[keep], iu[keep]] = True
    W[iu[keep], ju[keep]] = vals[keep]
    W[ju[keep], iu[keep]] = vals[keep]
    return A, W

A_struct_und = ((C_strength + C_strength.T) > 0).toarray().astype(bool)
np.fill_diagonal(A_struct_und, False)
struct_degree_und = A_struct_und.sum(axis=1)
struct_edges_und = int(A_struct_und[np.triu_indices(N, k=1)].sum())
possible_edges = N * (N - 1) // 2
struct_density_und = struct_edges_und / possible_edges
struct_clustering = nx.average_clustering(nx.from_numpy_array(A_struct_und))

rows = []
for mult in [0.5, 1.0, 2.0, 4.0]:
    k = max(1, int(round(struct_edges_und * mult)))
    A_func, W_func = top_k_functional_adjacency(F_corr, k, valid_mask=valid, positive_only=True)
    func_degree = A_func.sum(axis=1)
    func_strength = np.nansum(np.where(A_func, W_func, 0.0), axis=1)
    rho_deg, p_deg = spearmanr(struct_degree_und[valid], func_degree[valid])
    rho_str, p_str = spearmanr(hub_metrics.loc[valid, 'total_strength'], func_strength[valid])
    func_clustering = nx.average_clustering(nx.from_numpy_array(A_func))
    rows.append({
        'density_multiplier': mult,
        'functional_edges': int(A_func[np.triu_indices(N, k=1)].sum()),
        'functional_density': float(A_func[np.triu_indices(N, k=1)].sum() / possible_edges),
        'structural_edges_undirected': struct_edges_und,
        'structural_density_undirected': struct_density_und,
        'spearman_struct_degree_vs_functional_degree': float(rho_deg),
        'p_degree': float(p_deg),
        'spearman_struct_strength_vs_functional_strength': float(rho_str),
        'p_strength': float(p_str),
        'structural_clustering': float(struct_clustering),
        'functional_clustering': float(func_clustering),
    })

topology = pd.DataFrame(rows)
out_topology = OUT_TBL / 'h7_topology_sensitivity.csv'
topology.to_csv(out_topology, index=False)
print(f'saved: {out_topology}')
topology


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

ax = axes[0]
ax.plot(
    topology.density_multiplier,
    topology.spearman_struct_degree_vs_functional_degree,
    marker='o',
    lw=2,
    label='degree rank coupling',
)
ax.plot(
    topology.density_multiplier,
    topology.spearman_struct_strength_vs_functional_strength,
    marker='s',
    lw=2,
    label='strength rank coupling',
)
ax.axhline(0, color='0.25', lw=0.8)
ax.set(xlabel='functional density / structural density', ylabel='Spearman rho', title='Thresholded topology coupling')
ax.legend(frameon=False)

ax = axes[1]
ax.axhline(struct_clustering, color='#4E79A7', ls='--', lw=1.8, label='structural clustering')
ax.plot(
    topology.density_multiplier,
    topology.functional_clustering,
    color='#E15759',
    marker='o',
    lw=2,
    label='functional clustering',
)
ax.set(xlabel='functional density / structural density', ylabel='average clustering', title='Clustering is threshold-sensitive')
ax.legend(frameon=False)

plt.tight_layout()
out_fig = OUT_FIG / 'h7_topology_sensitivity.png'
fig.savefig(out_fig, bbox_inches='tight')
print(f'saved: {out_fig}')
plt.show()


### Topology Conclusion

Topology similarity is weaker and more definition-dependent than the pair-level H1 result. Thresholded functional-network metrics depend on the chosen density and on whether signed, positive-only, or absolute correlations are used. This sensitivity is not a failure of the analysis; it reflects a real conceptual mismatch between sparse directed synapses and dense symmetric correlations.

The safest synthesis is that anatomical connectivity carries a clear pair-level functional signal, while broader topology matching is limited and exploratory. This directly answers the assignment prompt: connected neurons are more correlated on average, but the structural and functional networks should not be described as having strongly similar topology without substantial qualification.


## 9. Current Summary Table

This table pulls the current H1-H7 results into one report-facing summary. It is marked current because Person 4's 93-neuron sanity check and V1/RL/AL exploratory comparison still need to be appended.

After Person 4's section is added, this table should be extended rather than replaced. The most useful final version will keep the H1-H7 rows and add validation rows for the 93-neuron same-scan analysis and the area-specific exploratory checks.


In [ ]:
final_rows = []

final_rows.append({
    'analysis': 'H1',
    'cohort': '906',
    'question': 'Are directly connected pairs more functionally correlated?',
    'effect': H1_RESULT.get('mean_diff'),
    'ci_low': H1_RESULT.get('ci_lo'),
    'ci_high': H1_RESULT.get('ci_hi'),
    'statistic': H1_RESULT.get('perm_z'),
    'p_value': H1_RESULT.get('perm_p_one_sided'),
    'interpretation_short': 'Supported: connected pairs have a modest but robustly higher signal correlation.',
})
final_rows.append({
    'analysis': 'H2',
    'cohort': '906 connected pairs',
    'question': 'Does synapse strength grade functional similarity?',
    'effect': H2_RESULT.get('spearman_rho_sum'),
    'ci_low': H2_RESULT.get('spearman_ci_lo'),
    'ci_high': H2_RESULT.get('spearman_ci_hi'),
    'statistic': H2_RESULT.get('perm_z'),
    'p_value': H2_RESULT.get('perm_p_two_sided'),
    'interpretation_short': 'Supported but weak: stronger connected pairs are only slightly more correlated.',
})
final_rows.append({
    'analysis': 'H3',
    'cohort': '906',
    'question': 'Are reciprocal pairs more functionally similar than unidirectional pairs?',
    'effect': H3_RESULT.get('mean_bi') - H3_RESULT.get('mean_uni'),
    'ci_low': np.nan,
    'ci_high': np.nan,
    'statistic': H3_RESULT.get('perm_z_bi_gt_uni'),
    'p_value': H3_RESULT.get('perm_p_bi_gt_uni'),
    'interpretation_short': 'Partially supported: reciprocal pairs exceed unconnected pairs, but bi > uni is not detected.',
})

h4h6_summary = pd.DataFrame(RESULTS)
for hyp, preferred_test, short in [
    ('H4', 'distance-matched', 'Supported: the connected-pair effect shrinks under distance matching but remains positive.'),
    ('H5', 'joint logistic ?(f_corr | dist, area, layer, celltype)', 'Supported: f_corr remains positive after coarse composition controls.'),
    ('H6', 'joint logistic ?(f_corr | ori_sim, dist)', 'Supported with caveat: f_corr remains positive; orientation adds a smaller residual bias.'),
]:
    row = h4h6_summary[(h4h6_summary.hypothesis == hyp) & (h4h6_summary.test == preferred_test)]
    if len(row):
        row = row.iloc[0]
        final_rows.append({
            'analysis': hyp,
            'cohort': '906',
            'question': preferred_test,
            'effect': row.mean_diff,
            'ci_low': row.ci_lo,
            'ci_high': row.ci_hi,
            'statistic': np.nan,
            'p_value': row.p,
            'interpretation_short': short,
        })

h7_primary = h7_summary[
    (h7_summary.cohort == 'all_906')
    & (h7_summary.functional_metric == 'mean_fcorr_signed')
    & (h7_summary.structural_metric == 'total_strength')
    & (h7_summary.controls == 'distance+area')
]
if len(h7_primary):
    row = h7_primary.iloc[0]
    final_rows.append({
        'analysis': 'H7 / topology',
        'cohort': '906',
        'question': 'Are structural hubs also functional hubs?',
        'effect': row.rho,
        'ci_low': np.nan,
        'ci_high': np.nan,
        'statistic': row.rank_variance_approx,
        'p_value': row.p_value,
        'interpretation_short': 'Limited support: narrow signed hub coupling is weak and sensitivity-dependent.',
    })

if 'topology' in globals() and len(topology):
    topo_row = topology.loc[(topology.density_multiplier - 1.0).abs().idxmin()]
    final_rows.append({
        'analysis': 'thresholded topology',
        'cohort': '906',
        'question': 'Do structural and functional graphs have similar topology at matched density?',
        'effect': topo_row.spearman_struct_degree_vs_functional_degree,
        'ci_low': np.nan,
        'ci_high': np.nan,
        'statistic': topo_row.functional_clustering,
        'p_value': topo_row.p_degree,
        'interpretation_short': 'Exploratory: topology similarity depends strongly on functional-threshold definition.',
    })

final_summary = pd.DataFrame(final_rows)
out_final = TBL_DIR / 'final_analysis_summary_current.csv'
final_summary.to_csv(out_final, index=False)
print(f'saved: {out_final}')
final_summary


## 10. Pending Person 4 Section

Add Person 4's code here when available.

Planned additions:

- 93-neuron scan 9_3 sanity check.
- Optional true noise-correlation analysis if residual trial-level data are implemented.
- V1/RL/AL exploratory comparison.
- Any necessary updates to the final summary table and final conclusion.

This section should be written as validation and scope extension. It should not force the earlier conclusions to overclaim. If the 93-neuron analysis agrees with H1, it strengthens the main pair-level result. If it differs, the final discussion should explain whether that difference is due to same-scan measurement, smaller sample size, V1-only restriction, or the distinction between signal, trace, and noise correlation.


## 11. Provisional Overall Conclusion

The current merged analysis supports the assignment's main pair-level answer: directly connected neurons are more functionally similar than unconnected neurons in the 906-neuron cohort. The effect is statistically robust but quantitatively modest, survives distance and coarse composition controls, and is not replaced by orientation similarity.

The supporting hypotheses refine that conclusion. Synapse strength adds only a weak graded component, and reciprocity does not add a detectable premium beyond connectedness. These results suggest that anatomy carries functional information, but only noisily: direct connectivity shifts the expected functional similarity distribution rather than determining the functional relationship of individual pairs.

The network-level answer is more cautious. Structural-functional topology similarity is much less decisive than the H1 pair-level result. Signed hub coupling is weak, and thresholded graph comparisons are sensitive to how the dense functional matrix is converted into a graph. The final report should therefore separate two claims: direct structural connectivity predicts higher functional similarity, while broader structural-functional topology matching is limited and exploratory.

This conclusion should be updated after the pending Person 4 section is appended, especially if the 93-neuron same-scan sanity check or V1/RL/AL comparisons materially change the interpretation.
